# VLSI Physical Design - Complete Pipeline
# Step-by-Step Guide: From Data to GDS

## What This Notebook Does:

This notebook takes you through the **complete VLSI physical design flow** using Machine Learning:

1. **Setup & Installation** - Install required packages
2. **Data Loading** - Load CircuitNet dataset
3. **Data Understanding** - Explore and visualize the data
4. **Model Architecture** - Build GNN models
5. **Training** - Train placement prediction model
6. **Evaluation** - Test and visualize results
7. **Industry Visualization** - Micron-precision layouts
8. **Advanced Models** - Routing, congestion, timing
9. **GDS Export** - Integration with EDA tools
10. **Memory Cleanup** - Resource management

---

## Prerequisites:

- Python 3.8+
- 16GB+ RAM (32GB recommended)
- GPU with 8GB+ VRAM (optional but recommended)
- CircuitNet dataset downloaded




---

# STEP 1: Setup & Installation

Install all required packages for VLSI physical design with ML.

In [2]:
# Install PyTorch first (CUDA 11.8)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 

# Install PyTorch Geometric and extensions using pre-built wheels
!pip install torch-geometric 
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.7.0+cu118.html 

# Install other utilities
!pip install matplotlib numpy scipy pandas psutil 

print("All packages installed successfully!")

Looking in indexes: https://download.pytorch.org/whl/cu118



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in links: https://data.pyg.org/whl/torch-2.7.0+cu118.html



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


All packages installed successfully!



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

# STEP 2: Import Libraries

Import all necessary libraries for the pipeline.

In [3]:
# Core libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# PyTorch Geometric
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, GCNConv
from torch_geometric.loader import DataLoader

# Numerical and visualization
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle

# System utilities
import os
import time
import gc
import psutil
from scipy.spatial import KDTree
from collections import defaultdict, Counter

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"   PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\nAll libraries imported successfully!")

c:\Users\Admin\VLSI-\Ayush\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
   PyTorch version: 2.7.1+cu118
   GPU: NVIDIA GeForce RTX 4060
   GPU Memory: 8.59 GB

All libraries imported successfully!


---

# STEP 3: Configure Paths

Set up paths to your CircuitNet dataset.

In [ ]:
# ============================================
# CONFIGURE YOUR CIRCUITNET PATHS HERE
# ============================================

CIRCUITNET_BASE = r"C:\Users\Admin\VLSI-\Ayush\CircuitNet"  # <-- Set this to your CircuitNet base directory
model_save = r"C:\Users\Admin\VLSI-\Ayush\examples\vlsi_placement_model.pth"  # <-- Path to save/load your model
output_dir = r"C:\Users\Admin\VLSI-\Ayush\vlsi_results"  # <-- Path to save output results


# Data paths
PLACEMENT_PATH = os.path.join(CIRCUITNET_BASE, "instance_placement_micron-002", "instance_placement_micron")
NODE_ATTR_PATH = os.path.join(CIRCUITNET_BASE, "graph_features", "graph_information", "node_attr")
NET_ATTR_PATH = os.path.join(CIRCUITNET_BASE, "graph_features", "graph_information", "net_attr")
PIN_ATTR_PATH = os.path.join(CIRCUITNET_BASE, "graph_features", "graph_information", "pin_attr")
CONGESTION_PATH = os.path.join(CIRCUITNET_BASE, "congestion")

# Verify paths exist
print("Verifying dataset paths...\n")

paths_to_check = {
    "Base Directory": CIRCUITNET_BASE,
    "Placement Data": PLACEMENT_PATH,
    "Node Attributes": NODE_ATTR_PATH,
    "Net Attributes": NET_ATTR_PATH,
    "Pin Attributes": PIN_ATTR_PATH,
}

all_exist = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "[OK]" if exists else "[MISSING]"
    print(f"{status} {name}: {path}")
    if not exists:
        all_exist = False

print()
if all_exist:
    print("All paths verified! Ready to load data.")
else:
    print("WARNING: Some paths are missing. Please check your CircuitNet installation.")
    print("\nDownload from: https://drive.google.com/drive/folders/1GjW-1LBx1563bg3pHQGvhcEyK2A9sYUB")

Verifying dataset paths...

[MISSING] Base Directory: H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet
[MISSING] Placement Data: H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\instance_placement_micron-002\instance_placement_micron
[MISSING] Node Attributes: H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\graph_features\graph_information\node_attr
[MISSING] Net Attributes: H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\graph_features\graph_information\net_attr
[MISSING] Pin Attributes: H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\graph_features\graph_information\pin_attr


Download from: https://drive.google.com/drive/folders/1GjW-1LBx1563bg3pHQGvhcEyK2A9sYUB


---

# STEP 4: Define Data Loaders

Functions to load and process CircuitNet data.

In [5]:
def get_design_name_from_placement(filename):
    """
    Extract base design name from placement filename
    e.g., '7873-zero-riscy-a-1-c20-u0.9-m1-p2-f1.npy' -> 'zero-riscy-a-1-c20'
    """
    name = os.path.splitext(filename)[0]
    parts = name.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    
    design_parts = []
    for p in parts:
        design_parts.append(p)
        if p.startswith('c') and p[1:].isdigit():
            break
    return '-'.join(design_parts)


def build_netlist_edges(pin_attr, node_attr, placement_cell_names, star_threshold=10):
    """
    Build real netlist-based edges from CircuitNet pin_attr data.
    
    pin_attr structure (3 x num_pins):
        row[0] = pin name (str)
        row[1] = net index (int)  — which net this pin belongs to
        row[2] = node index (int) — which cell (in node_attr ordering) this pin belongs to
    
    node_attr structure (2 x num_cells_in_design):
        row[0] = cell names
        row[1] = cell types
    
    Also returns per-cell connectivity stats for feature engineering.
    
    Returns:
        edge_index: np.array of shape [2, num_edges], dtype int64
        cell_connectivity: dict mapping placement_idx -> {pin_count, net_count, avg_fanout, max_fanout}
    """
    num_placement_cells = len(placement_cell_names)
    
    # Map: node_attr cell name → node_attr index
    nodeattr_names = list(node_attr[0])
    nodeattr_name_to_idx = {name: i for i, name in enumerate(nodeattr_names)}
    
    # Map: placement cell name → placement index
    pl_name_to_idx = {name: i for i, name in enumerate(placement_cell_names)}
    
    # Map: node_attr index → placement index (only for cells present in both)
    nodeattr_to_pl = {}
    for na_name, na_idx in nodeattr_name_to_idx.items():
        if na_name in pl_name_to_idx:
            nodeattr_to_pl[na_idx] = pl_name_to_idx[na_name]
    
    # ============================================================
    # Group pins by net AND track per-cell connectivity
    # ============================================================
    net_to_pl_cells = defaultdict(set)
    cell_pin_count = defaultdict(int)      # pins per cell
    cell_net_set = defaultdict(set)        # nets per cell
    
    for pin_i in range(pin_attr.shape[1]):
        raw_net = pin_attr[1][pin_i]
        raw_node = pin_attr[2][pin_i]
        
        # Handle node index (could be int or list)
        na_idx = int(raw_node) if not isinstance(raw_node, (list, np.ndarray)) else int(raw_node[0])
        pl_idx = nodeattr_to_pl.get(na_idx)
        if pl_idx is None:
            continue
        
        # Handle net index (could be int or list of ints for multi-net pins)
        if isinstance(raw_net, (list, np.ndarray)):
            net_indices = [int(n) for n in raw_net]
        else:
            net_indices = [int(raw_net)]
        
        cell_pin_count[pl_idx] += 1
        for net_idx in net_indices:
            net_to_pl_cells[net_idx].add(pl_idx)
            cell_net_set[pl_idx].add(net_idx)
    
    # Build edges from nets
    edge_set = set()
    for net_idx, cells in net_to_pl_cells.items():
        cells = sorted(cells)
        if len(cells) < 2:
            continue
        if len(cells) <= star_threshold:
            for i in range(len(cells)):
                for j in range(i + 1, len(cells)):
                    edge_set.add((cells[i], cells[j]))
                    edge_set.add((cells[j], cells[i]))
        else:
            hub = cells[0]
            for c in cells[1:]:
                edge_set.add((hub, c))
                edge_set.add((c, hub))
    
    # Track which cells got netlist edges
    cells_with_edges = set()
    for s, d in edge_set:
        cells_with_edges.add(s)
        cells_with_edges.add(d)
    
    orphan_cells = [i for i in range(num_placement_cells) if i not in cells_with_edges]
    
    if edge_set:
        edges = np.array(sorted(edge_set), dtype=np.int64).T
    else:
        edges = np.zeros((2, 0), dtype=np.int64)
    
    # ============================================================
    # Build per-cell connectivity features
    # ============================================================
    cell_connectivity = {}
    for pl_idx in range(num_placement_cells):
        pin_cnt = cell_pin_count.get(pl_idx, 0)
        nets = cell_net_set.get(pl_idx, set())
        net_cnt = len(nets)
        
        # Average and max fanout of nets this cell belongs to
        fanouts = [len(net_to_pl_cells[n]) for n in nets] if nets else [0]
        avg_fanout = np.mean(fanouts) if fanouts else 0
        max_fanout = max(fanouts) if fanouts else 0
        
        cell_connectivity[pl_idx] = {
            'pin_count': pin_cnt,
            'net_count': net_cnt,
            'avg_fanout': avg_fanout,
            'max_fanout': max_fanout,
        }
    
    return edges, len(orphan_cells), len(net_to_pl_cells), cell_connectivity


# ============================================================
# MACRO CELL TYPE CATEGORIES (industry-standard classification)
# ============================================================
MACRO_TYPES = {'PLL', 'SRAM', 'RAM', 'ROM', 'PAD', 'IO', 'ANALOG', 'DSP', 'FIFO'}

def classify_cell(cell_type, cell_area, area_threshold=100):
    """
    Classify cell as macro, standard, or filler based on type and area.
    Returns: (is_macro, is_filler, category_id)
        category_id: 0=filler/tap, 1=std_combo, 2=std_seq, 3=macro
    """
    ct = cell_type.upper()
    
    # Check if it's a known macro type
    for mt in MACRO_TYPES:
        if mt in ct:
            return True, False, 3
    
    # Large area = macro
    if cell_area > area_threshold:
        return True, False, 3
    
    # Filler / tap cells (very small, zero area)
    if cell_area < 0.01 or 'FILL' in ct or 'TAP' in ct or 'TIE' in ct or 'DCAP' in ct:
        return False, True, 0
    
    # Sequential cells (flip-flops, latches)
    if any(k in ct for k in ['DF', 'LATCH', 'FF', 'REG', 'DL']):
        return False, False, 2
    
    # Standard combinational
    return False, False, 1


def load_circuitnet_sample(placement_file):
    """
    Load a single CircuitNet sample with INDUSTRY-RELEVANT features.
    
    Enhanced Feature Vector (16 features per cell):
    ─────────────────────────────────────────────────
    [0]  x_position      - Normalized X position
    [1]  y_position      - Normalized Y position
    [2]  wi          - Normalized cell width
    [3dth ]  height          - Normalized cell height
    [4]  log_area        - Log of cell area (size importance)
    [5]  cell_type_enc   - Encoded cell type (0-1)
    [6]  is_macro        - 1 if macro (PLL, SRAM, etc), 0 otherwise
    [7]  cell_category   - Category: 0=filler, 1=combo, 2=seq, 3=macro
    [8]  pin_count       - Number of pins (connectivity importance)
    [9]  net_count       - Number of nets connected (connectivity degree)
    [10] avg_net_fanout  - Avg fanout of connected nets (spread importance)
    [11] max_net_fanout  - Max fanout (critical net detection)
    [12] relative_area   - Cell area / total chip area (size significance)
    [13] aspect_ratio    - Width / Height ratio
    [14] neighbors_size  - Avg size of connected neighbors (context)
    [15] connectivity_importance - pin_count * net_count (placement priority)
    """
    # Load placement data
    placement_path = os.path.join(PLACEMENT_PATH, placement_file)
    placement_raw = np.load(placement_path, allow_pickle=True)
    placement_dict = placement_raw.item() if hasattr(placement_raw, 'item') else placement_raw
    
    # Get design name
    design_name = get_design_name_from_placement(placement_file)
    
    # Load node attributes
    node_attr_file = os.path.join(NODE_ATTR_PATH, f"{design_name}_node_attr.npy")
    node_attr = None
    if os.path.exists(node_attr_file):
        try:
            node_attr = np.load(node_attr_file, allow_pickle=True)
        except:
            node_attr = None
    
    # Load pin attributes (for netlist connectivity)
    pin_attr_file = os.path.join(PIN_ATTR_PATH, f"{design_name}_pin_attr.npy")
    pin_attr = None
    if os.path.exists(pin_attr_file):
        try:
            pin_attr = np.load(pin_attr_file, allow_pickle=True)
        except:
            pin_attr = None
    
    # Extract cell information
    cell_names = list(placement_dict.keys())
    num_nodes = len(cell_names)
    
    # Extract coordinates and sizes
    coords = []
    sizes = []
    areas = []
    for name in cell_names:
        bbox = placement_dict[name]  # [x_min, y_min, x_max, y_max]
        x_center = (bbox[0] + bbox[2]) / 2
        y_center = (bbox[1] + bbox[3]) / 2
        width = bbox[2] - bbox[0]
        height = bbox[3] - bbox[1]
        area = width * height
        coords.append([x_center, y_center])
        sizes.append([width, height])
        areas.append(area)
    
    coords = np.array(coords, dtype=np.float32)
    sizes = np.array(sizes, dtype=np.float32)
    areas = np.array(areas, dtype=np.float32)
    total_chip_area = (coords[:, 0].max() - coords[:, 0].min()) * (coords[:, 1].max() - coords[:, 1].min()) + 1e-8
    
    # Normalize coordinates to [0, 1]
    coords_norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    sizes_norm = sizes / (sizes.max() + 1e-8)
    
    # ==========================================
    # Build cell type and macro classification
    # ==========================================
    cell_types_ordered = []
    is_macro = np.zeros(num_nodes, dtype=np.float32)
    is_filler = np.zeros(num_nodes, dtype=np.float32)
    cell_category = np.zeros(num_nodes, dtype=np.float32)
    type_encoding = np.zeros(num_nodes, dtype=np.float32)
    
    if node_attr is not None and node_attr.shape[0] == 2:
        na_names = list(node_attr[0])
        na_types = list(node_attr[1])
        name_to_type = {n: t for n, t in zip(na_names, na_types)}
        cell_types_ordered = [name_to_type.get(cn, "UNKNOWN") for cn in cell_names]
        unique_types = sorted(set(cell_types_ordered))
        type_to_idx = {t: i for i, t in enumerate(unique_types)}
        type_enc = np.array([type_to_idx[t] for t in cell_types_ordered], dtype=np.float32)
        if len(unique_types) > 1:
            type_encoding = type_enc / (len(unique_types) - 1)
        
        for i, (ct, a) in enumerate(zip(cell_types_ordered, areas)):
            macro, filler, cat = classify_cell(ct, a)
            is_macro[i] = float(macro)
            is_filler[i] = float(filler)
            cell_category[i] = cat / 3.0  # normalize to [0, 1]
    else:
        cell_types_ordered = ["UNKNOWN"] * num_nodes
        # Classify by area alone
        for i, a in enumerate(areas):
            if a > 100:
                is_macro[i] = 1.0
                cell_category[i] = 1.0
    
    # ==========================================
    # Build REAL netlist connectivity from pin_attr
    # ==========================================
    use_netlist = (pin_attr is not None and node_attr is not None)
    cell_connectivity = {}
    
    if use_netlist:
        edge_index, n_orphans, n_nets, cell_connectivity = build_netlist_edges(
            pin_attr, node_attr, cell_names, star_threshold=10
        )
        if edge_index.shape[1] == 0:
            use_netlist = False
    
    if not use_netlist:
        print(f"   WARNING: No pin_attr for {design_name}, falling back to KNN edges")
        k_neighbors = min(8, num_nodes - 1)
        if num_nodes > 1:
            tree = KDTree(coords_norm)
            _, indices = tree.query(coords_norm, k=k_neighbors + 1)
            src = np.repeat(np.arange(num_nodes), k_neighbors)
            dst = indices[:, 1:].flatten()
            edge_index = np.stack([src, dst], axis=0).astype(np.int64)
        else:
            edge_index = np.array([[0], [0]], dtype=np.int64)
        # Default connectivity for all cells
        for i in range(num_nodes):
            cell_connectivity[i] = {'pin_count': 0, 'net_count': 0, 'avg_fanout': 0, 'max_fanout': 0}
    
    # ==========================================
    # Extract connectivity features per cell
    # ==========================================
    pin_counts = np.array([cell_connectivity.get(i, {}).get('pin_count', 0) for i in range(num_nodes)], dtype=np.float32)
    net_counts = np.array([cell_connectivity.get(i, {}).get('net_count', 0) for i in range(num_nodes)], dtype=np.float32)
    avg_fanouts = np.array([cell_connectivity.get(i, {}).get('avg_fanout', 0) for i in range(num_nodes)], dtype=np.float32)
    max_fanouts = np.array([cell_connectivity.get(i, {}).get('max_fanout', 0) for i in range(num_nodes)], dtype=np.float32)
    
    # Normalize connectivity features
    pin_counts_norm = pin_counts / (pin_counts.max() + 1e-8)
    net_counts_norm = net_counts / (net_counts.max() + 1e-8)
    avg_fanouts_norm = avg_fanouts / (avg_fanouts.max() + 1e-8)
    max_fanouts_norm = max_fanouts / (max_fanouts.max() + 1e-8)
    
    # Relative area (cell area / total chip area)
    relative_area = areas / (total_chip_area + 1e-8)
    relative_area_norm = relative_area / (relative_area.max() + 1e-8)
    
    # Aspect ratio
    aspect_ratio = sizes[:, 0] / (sizes[:, 1] + 1e-8)
    aspect_ratio = np.clip(aspect_ratio, 0, 10)
    aspect_ratio_norm = aspect_ratio / (aspect_ratio.max() + 1e-8)
    
    # Connectivity importance = pin_count * net_count (higher = more critical to place well)
    conn_importance = pin_counts * net_counts
    conn_importance_norm = conn_importance / (conn_importance.max() + 1e-8)
    
    # Average neighbor size (computed from edges)
    neighbor_area_avg = np.zeros(num_nodes, dtype=np.float32)
    if edge_index.shape[1] > 0:
        ei = edge_index  # [2, E]
        for e in range(ei.shape[1]):
            src, dst = ei[0, e], ei[1, e]
            neighbor_area_avg[src] += areas[dst]
        # Average by degree
        degree = np.bincount(ei[0], minlength=num_nodes).astype(np.float32)
        degree[degree == 0] = 1
        neighbor_area_avg /= degree
    neighbor_area_norm = neighbor_area_avg / (neighbor_area_avg.max() + 1e-8)
    
    # ==========================================
    # Assemble 16-feature vector
    # ==========================================
    NUM_FEATURES = 16
    node_features = np.zeros((num_nodes, NUM_FEATURES), dtype=np.float32)
    node_features[:, 0]  = coords_norm[:, 0]       # x position
    node_features[:, 1]  = coords_norm[:, 1]       # y position
    node_features[:, 2]  = sizes_norm[:, 0]        # width
    node_features[:, 3]  = sizes_norm[:, 1]        # height
    node_features[:, 4]  = np.log1p(areas)         # log area
    node_features[:, 4] /= (node_features[:, 4].max() + 1e-8)  # normalize
    node_features[:, 5]  = type_encoding           # cell type encoded
    node_features[:, 6]  = is_macro                # is_macro flag
    node_features[:, 7]  = cell_category           # cell category (0-1)
    node_features[:, 8]  = pin_counts_norm         # pin count (connectivity)
    node_features[:, 9]  = net_counts_norm         # net count (connectivity)
    node_features[:, 10] = avg_fanouts_norm        # avg fanout of connected nets
    node_features[:, 11] = max_fanouts_norm        # max fanout
    node_features[:, 12] = relative_area_norm      # relative area
    node_features[:, 13] = aspect_ratio_norm       # aspect ratio
    node_features[:, 14] = neighbor_area_norm      # avg neighbor size
    node_features[:, 15] = conn_importance_norm    # connectivity importance score
    
    # Convert to PyTorch tensors
    data = Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        y=torch.tensor(coords_norm, dtype=torch.float)
    )
    data.num_cells = num_nodes
    data.sample_name = os.path.splitext(placement_file)[0]
    data.design_name = design_name
    data.original_coords = torch.tensor(coords, dtype=torch.float)
    data.is_macro = torch.tensor(is_macro, dtype=torch.float)
    data.areas = torch.tensor(areas, dtype=torch.float)
    
    return data


def load_circuitnet_dataset(max_samples=100):
    """
    Load multiple CircuitNet samples as a dataset
    """
    samples = [f for f in os.listdir(PLACEMENT_PATH) if f.endswith('.npy')][:max_samples]
    
    if not samples:
        print("No samples found!")
        return None
    
    dataset = []
    errors = 0
    print(f"Loading {len(samples)} CircuitNet samples...")
    
    for i, sample_file in enumerate(samples):
        try:
            data = load_circuitnet_sample(sample_file)
            dataset.append(data)
            if (i + 1) % 50 == 0:
                print(f"   Loaded {i+1}/{len(samples)} samples... ({len(dataset)} successful, {errors} errors)")
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"   WARNING: Error loading {sample_file}: {str(e)[:80]}")
    
    if errors > 3:
        print(f"   ... and {errors - 3} more errors (suppressed)")
    
    if dataset:
        s = dataset[0]
        print(f"\n   Connectivity: netlist-based edges")
        print(f"   Sample '{s.sample_name}': {s.num_cells:,} cells, {s.edge_index.shape[1]:,} edges")
        avg_degree = s.edge_index.shape[1] / s.num_cells
        print(f"   Average degree: {avg_degree:.1f} edges per cell")
        
        # Show macro summary
        n_macros = int(s.is_macro.sum())
        print(f"   Macros detected: {n_macros} (vs {s.num_cells - n_macros:,} standard cells)")
        print(f"   Feature vector: {s.x.shape[1]} dimensions per cell")
    
    print(f"\nLoaded {len(dataset)} samples successfully! ({errors} failed)")
    return dataset


# Feature name reference for inspection
FEATURE_NAMES = [
    "[0]  X position (normalized)",
    "[1]  Y position (normalized)",
    "[2]  Cell width (normalized)",
    "[3]  Cell height (normalized)",
    "[4]  Log area (normalized)",
    "[5]  Cell type (encoded)",
    "[6]  Is macro (PLL/SRAM/etc)",
    "[7]  Cell category (filler/combo/seq/macro)",
    "[8]  Pin count (connectivity degree)",
    "[9]  Net count (nets connected)",
    "[10] Avg net fanout (spread)",
    "[11] Max net fanout (critical net)",
    "[12] Relative area (cell/chip)",
    "[13] Aspect ratio (w/h)",
    "[14] Avg neighbor size (context)",
    "[15] Connectivity importance (pins×nets)",
]

print("Data loader functions defined!")
print("   - 16 INDUSTRY-RELEVANT features per cell:")
print("     Size features: width, height, area, relative_area, aspect_ratio")
print("     Connectivity features: pin_count, net_count, fanout, importance")
print("     Classification: is_macro, cell_category, cell_type")
print("     Context: neighbor_size, position")
print("   - Real netlist connectivity from CircuitNet pin_attr")
print("   - Macro detection (PLL, SRAM, etc.)")

Data loader functions defined!
   - 16 INDUSTRY-RELEVANT features per cell:
     Size features: width, height, area, relative_area, aspect_ratio
     Connectivity features: pin_count, net_count, fanout, importance
     Classification: is_macro, cell_category, cell_type
     Context: neighbor_size, position
   - Real netlist connectivity from CircuitNet pin_attr
   - Macro detection (PLL, SRAM, etc.)


---

# STEP 5: Load Dataset

Load CircuitNet samples into memory.

**Adjust `max_samples` based on your RAM:**
- 16GB RAM: Use 100-200 samples
- 32GB RAM: Use 500-1000 samples
- 64GB RAM: Use 2000+ samples

In [6]:
# Load dataset
MAX_SAMPLES = 150 # Increased from 100 for better generalization

print(f"Loading {MAX_SAMPLES} samples from CircuitNet...\n")
circuitnet_dataset = load_circuitnet_dataset(max_samples=MAX_SAMPLES)

if circuitnet_dataset:
    # Split into train/test (80/20)
    split_idx = int(len(circuitnet_dataset) * 0.8)
    cn_train = circuitnet_dataset[:split_idx]
    cn_test = circuitnet_dataset[split_idx:]
    
    print(f"\nDataset Statistics:")
    print(f"   Total samples: {len(circuitnet_dataset)}")
    print(f"   Training samples: {len(cn_train)}")
    print(f"   Test samples: {len(cn_test)}")
    print(f"   Cells per sample: ~{circuitnet_dataset[0].num_cells:,}")
    print(f"   Edges per sample: ~{circuitnet_dataset[0].edge_index.shape[1]:,}")
    print(f"\nDataset ready for training!")
else:
    print("Failed to load dataset. Check your paths!")

Loading 150 samples from CircuitNet...



FileNotFoundError: [WinError 3] The system cannot find the path specified: 'H:\\Labs\\Generative Ai\\Ayush1\\Ayush\\CircuitNet\\instance_placement_micron-002\\instance_placement_micron'

---

# STEP 6: Explore Sample Data

Let's look at a single sample to understand the data structure.

In [ ]:
# Explore a sample
if circuitnet_dataset:
    sample = circuitnet_dataset[0]
    
    print("Sample Data Structure:\n")
    print(f"Sample Name: {sample.sample_name}")
    print(f"Design: {sample.design_name}")
    print(f"Number of cells: {sample.num_cells:,}")
    print(f"Number of edges: {sample.edge_index.shape[1]:,}")
    print(f"Macros: {int(sample.is_macro.sum())} | Std cells: {sample.num_cells - int(sample.is_macro.sum()):,}")
    print(f"\nNode features shape: {sample.x.shape}")
    print(f"   - Feature vector per cell: {sample.x.shape[1]} dimensions (industry-relevant)")
    print(f"\nTarget coordinates shape: {sample.y.shape}")
    print(f"   - (x, y) position per cell")
    print(f"\nEdge index shape: {sample.edge_index.shape}")
    print(f"   - Netlist-based connectivity (cells sharing nets)")
    
    print(f"\nAll 16 Feature Statistics:")
    print(f"{'#':<4} {'Feature':<40} {'Min':>8} {'Max':>8} {'Mean':>8} {'NonZero%':>9}")
    print(f"{'─'*4} {'─'*40} {'─'*8} {'─'*8} {'─'*8} {'─'*9}")
    for i in range(sample.x.shape[1]):
        col = sample.x[:, i].numpy()
        nz = (col != 0).sum() / len(col) * 100
        name = FEATURE_NAMES[i] if i < len(FEATURE_NAMES) else f"[{i}] Unknown"
        print(f"{i:<4} {name:<40} {col.min():>8.4f} {col.max():>8.4f} {col.mean():>8.4f} {nz:>8.1f}%")
    
    print("\nSample exploration complete!")

In [ ]:
# ============================================================
# INSPECT TRAINING DATA — What the model sees & learns from
# ============================================================

sample = cn_train[0]  # First training sample

print("=" * 80)
print("TRAINING DATA INSPECTION — INDUSTRY-RELEVANT FEATURES")
print("=" * 80)

# ---------- 1. Overall Structure ----------
print("\n OVERALL DATA STRUCTURE")
print(f"   Sample Name      : {sample.sample_name}")
print(f"   Design Name      : {sample.design_name}")
print(f"   Number of Cells  : {sample.num_cells:,}")
print(f"   Number of Edges  : {sample.edge_index.shape[1]:,}")
print(f"   Avg Degree       : {sample.edge_index.shape[1] / sample.num_cells:.1f} edges/cell")
n_macros = int(sample.is_macro.sum())
print(f"   Macros           : {n_macros} (PLL, SRAM, etc.)")
print(f"   Standard cells   : {sample.num_cells - n_macros:,}")

# ---------- 2. Input Features (X) — what the model RECEIVES ----------
print("\n" + "=" * 80)
print("INPUT FEATURES (data.x) — 16 Industry-Relevant Features per Cell")
print("=" * 80)
print(f"   Shape: {sample.x.shape}  →  ({sample.num_cells:,} cells × {sample.x.shape[1]} features)")
print()
print(f"   GROUP 1: POSITION (where is it?)")
print(f"   GROUP 2: SIZE (how big is it?)")
print(f"   GROUP 3: CONNECTIVITY (how many connections?)")
print(f"   GROUP 4: CLASSIFICATION (what type of cell?)")

print(f"\n   {'#':<4} {'Feature Name':<42} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8} {'NZ%':>7}")
print(f"   {'─'*4} {'─'*42} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*7}")

for i in range(sample.x.shape[1]):
    col = sample.x[:, i].numpy()
    nz_pct = (col != 0).sum() / len(col) * 100
    name = FEATURE_NAMES[i] if i < len(FEATURE_NAMES) else f"[{i}] Unknown"
    print(f"   {i:<4} {name:<42} {col.min():>8.4f} {col.max():>8.4f} {col.mean():>8.4f} {col.std():>8.4f} {nz_pct:>6.1f}%")

# ---------- 3. Target Labels (Y) — what the model PREDICTS ----------
print("\n" + "=" * 80)
print("TARGET LABELS (data.y) — What the GNN learns to predict")
print("=" * 80)
print(f"   Shape: {sample.y.shape}  →  ({sample.num_cells:,} cells × 2 coordinates)")
print(f"   The model predicts (X, Y) placement positions for each cell")

y_np = sample.y.numpy()
print(f"\n   {'Coord':<15} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
print(f"   {'─'*15} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
print(f"   {'X (normalized)':<15} {y_np[:,0].min():>10.4f} {y_np[:,0].max():>10.4f} {y_np[:,0].mean():>10.4f} {y_np[:,0].std():>10.4f}")
print(f"   {'Y (normalized)':<15} {y_np[:,1].min():>10.4f} {y_np[:,1].max():>10.4f} {y_np[:,1].mean():>10.4f} {y_np[:,1].std():>10.4f}")

# ---------- 4. Edge Index — connectivity ----------
print("\n" + "=" * 80)
print("EDGE INDEX (data.edge_index) — Graph connectivity")
print("=" * 80)
print(f"   Shape: {sample.edge_index.shape}  →  (2 × {sample.edge_index.shape[1]:,} directed edges)")
print(f"   Source: Real netlist connectivity from CircuitNet pin_attr")

ei = sample.edge_index.numpy()
degrees = np.bincount(ei[0], minlength=sample.num_cells)
print(f"\n   Degree Statistics:")
print(f"      Min degree    : {degrees.min()}")
print(f"      Max degree    : {degrees.max()}")
print(f"      Mean degree   : {degrees.mean():.1f}")
print(f"      Isolated cells: {(degrees == 0).sum():,} ({(degrees == 0).sum()/len(degrees)*100:.1f}%)")

# ---------- 5. Macro vs Standard Cell comparison ----------
print("\n" + "=" * 80)
print("MACRO vs STANDARD CELL COMPARISON (Industry Focus)")
print("=" * 80)

macro_mask = sample.is_macro.numpy().astype(bool)
std_mask = ~macro_mask

macro_x = sample.x[macro_mask].numpy()
std_x = sample.x[std_mask].numpy()

print(f"\n   {'Metric':<35} {'Macros':>12} {'Std Cells':>12} {'Ratio':>8}")
print(f"   {'─'*35} {'─'*12} {'─'*12} {'─'*8}")
print(f"   {'Count':<35} {macro_mask.sum():>12,} {std_mask.sum():>12,} {'':>8}")
if macro_mask.any():
    print(f"   {'Avg area (feat[4])':<35} {macro_x[:, 4].mean():>12.4f} {std_x[:, 4].mean():>12.4f} {macro_x[:, 4].mean()/(std_x[:, 4].mean()+1e-8):>7.1f}x")
    print(f"   {'Avg pin count (feat[8])':<35} {macro_x[:, 8].mean():>12.4f} {std_x[:, 8].mean():>12.4f} {macro_x[:, 8].mean()/(std_x[:, 8].mean()+1e-8):>7.1f}x")
    print(f"   {'Avg net count (feat[9])':<35} {macro_x[:, 9].mean():>12.4f} {std_x[:, 9].mean():>12.4f} {macro_x[:, 9].mean()/(std_x[:, 9].mean()+1e-8):>7.1f}x")
    print(f"   {'Avg connectivity importance (f[15])':<35} {macro_x[:, 15].mean():>12.4f} {std_x[:, 15].mean():>12.4f} {macro_x[:, 15].mean()/(std_x[:, 15].mean()+1e-8):>7.1f}x")

# ---------- 6. Sample cells ----------
print("\n" + "=" * 80)
print("SAMPLE CELLS — Macro cells and a few standard cells")
print("=" * 80)

# Show all macros first
macro_indices = np.where(macro_mask)[0]
for idx in macro_indices[:5]:
    feats = sample.x[idx].numpy()
    target = sample.y[idx].numpy()
    print(f"\n   [MACRO] Cell #{idx}:")
    print(f"      Size: w={feats[2]:.4f}, h={feats[3]:.4f}, area={feats[4]:.4f}, relative_area={feats[12]:.4f}")
    print(f"      Connectivity: pins={feats[8]:.4f}, nets={feats[9]:.4f}, importance={feats[15]:.4f}")
    print(f"      Target position: x={target[0]:.4f}, y={target[1]:.4f}")

# Show a few standard cells
std_indices = np.where(std_mask)[0][:3]
for idx in std_indices:
    feats = sample.x[idx].numpy()
    target = sample.y[idx].numpy()
    print(f"\n   [STD]   Cell #{idx}:")
    print(f"      Size: w={feats[2]:.4f}, h={feats[3]:.4f}, area={feats[4]:.4f}, relative_area={feats[12]:.4f}")
    print(f"      Connectivity: pins={feats[8]:.4f}, nets={feats[9]:.4f}, importance={feats[15]:.4f}")
    print(f"      Target position: x={target[0]:.4f}, y={target[1]:.4f}")

# ---------- 7. Feature distributions (grouped) ----------
print("\n" + "=" * 80)
print("FEATURE DISTRIBUTIONS BY CATEGORY")
print("=" * 80)

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i in range(min(16, sample.x.shape[1])):
    ax = axes[i]
    vals = sample.x[:, i].numpy()
    
    if macro_mask.any():
        ax.hist(vals[std_mask], bins=50, color='steelblue', alpha=0.7, label='Std cells', density=True)
        ax.hist(vals[macro_mask], bins=20, color='red', alpha=0.8, label='Macros', density=True)
        ax.legend(fontsize=7)
    else:
        ax.hist(vals, bins=50, color='steelblue', alpha=0.8, density=True)
    
    name = FEATURE_NAMES[i].split(']')[1].strip() if i < len(FEATURE_NAMES) else f"Feature {i}"
    ax.set_title(f'[{i}] {name}', fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'16 Industry Features — {sample.sample_name} ({sample.num_cells:,} cells, {n_macros} macros)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ---------- 8. Dataset summary ----------
print("\n" + "=" * 80)
print(f"DATASET SUMMARY — {len(cn_train)} training samples")
print("=" * 80)

all_cells = [s.num_cells for s in cn_train]
all_edges = [s.edge_index.shape[1] for s in cn_train]
all_macros = [int(s.is_macro.sum()) for s in cn_train]
all_designs = set(s.design_name for s in cn_train)

print(f"   Training samples  : {len(cn_train)}")
print(f"   Unique designs    : {len(all_designs)} → {sorted(all_designs)}")
print(f"   Cells per sample  : min={min(all_cells):,}, max={max(all_cells):,}, avg={np.mean(all_cells):,.0f}")
print(f"   Macros per sample : min={min(all_macros)}, max={max(all_macros)}, avg={np.mean(all_macros):.1f}")
print(f"   Edges per sample  : min={min(all_edges):,}, max={max(all_edges):,}, avg={np.mean(all_edges):,.0f}")
print(f"\n   Model trains on : {sample.x.shape[1]} features per cell + graph edges")
print(f"   Model predicts  : (x, y) position for each cell")
print(f"   Loss function   : MSE on positions")
print(f"\n   INDUSTRY INSIGHT: Macros have {np.mean([macro_x[:, 8].mean() for _ in [1]])/(std_x[:, 8].mean()+1e-8):.0f}x more connectivity")
print(f"   → Model learns to prioritize macro placement (like real EDA tools)")

print("\n" + "=" * 80)
print("INSPECTION COMPLETE")
print("=" * 80)

---

# STEP 7: Define GNN Model

Create the Graph Attention Network for placement prediction.

In [7]:
class VLSIPlacementGNN(nn.Module):
    """
    Graph Attention Network for VLSI cell placement prediction
    
    Industry-Aware Architecture (v2 - Anti-Collapse):
    - Input: 16 features per cell (size, connectivity, macro classification, context)
    - 4 GAT layers with multi-head attention + residual connections
    - NO Sigmoid bottleneck - uses scaled tanh for full [0,1] range utilization
    - Residual connections prevent gradient vanishing in deep GNN
    - Output: (x, y) coordinates for each cell
    """
    
    def __init__(self, input_dim=16, hidden_dim=128, output_dim=2, num_layers=4, heads=4):
        super(VLSIPlacementGNN, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers
        
        # Input projection
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        
        # GAT layers with attention + layer norms for residual connections
        self.gat_layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for i in range(num_layers):
            in_channels = hidden_dim
            out_channels = hidden_dim
            self.gat_layers.append(
                GATConv(in_channels, out_channels // heads, heads=heads, dropout=0.1, concat=True)
            )
            self.layer_norms.append(nn.LayerNorm(hidden_dim))
        
        # Output projection - no Sigmoid! Use clamped output instead
        # Sigmoid causes center collapse by squashing gradients at extremes
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        
        # Input projection
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = F.relu(x)
        
        # GAT layers with RESIDUAL connections
        for i, (gat_layer, layer_norm) in enumerate(zip(self.gat_layers, self.layer_norms)):
            residual = x
            x = gat_layer(x, edge_index)
            x = layer_norm(x)
            if i < len(self.gat_layers) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=0.1, training=self.training)
            # Residual connection - prevents information loss in deep GNN
            x = x + residual
        
        # Output projection
        out = self.output_proj(x)
        
        # Clamp to [0, 1] instead of Sigmoid
        # This preserves gradients at boundaries (Sigmoid kills them)
        out = out.clamp(0.0, 1.0)
        
        return out

# Create model with 16 input features (industry-relevant)
model = VLSIPlacementGNN(
    input_dim=16,      # 16 industry-relevant features
    hidden_dim=128,
    output_dim=2,
    num_layers=4,
    heads=4
).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Architecture (v2 - Anti-Collapse):\n")
print(model)
print(f"\nModel Statistics:")
print(f"   Total parameters: {num_params:,}")
print(f"   Input features: 16 dimensions (industry-relevant)")
print(f"   Key improvements over v1:")
print(f"     - Residual connections (prevents gradient vanishing)")
print(f"     - LayerNorm (stabilizes training)")
print(f"     - No Sigmoid (prevents center collapse)")
print(f"     - Clamp [0,1] output (preserves gradients at boundaries)")
print(f"   Device: {device}")
print("\nModel created successfully!")

Model Architecture (v2 - Anti-Collapse):

VLSIPlacementGNN(
  (input_proj): Linear(in_features=16, out_features=128, bias=True)
  (input_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (gat_layers): ModuleList(
    (0-3): 4 x GATConv(128, 32, heads=4)
  )
  (layer_norms): ModuleList(
    (0-3): 4 x LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (output_proj): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)

Model Statistics:
   Total parameters: 78,914
   Input features: 16 dimensions (industry-relevant)
   Key improvements over v1:
     - Residual connections (prevents gradient vanishing)
     - LayerNorm (stabilizes training)
     - No Sigmoid (prevents center collapse)
     - Clamp [0,1] output (preserves gradients at boundaries)
   Device: cuda

Model created successfully!


---

# STEP 8: Define Training Loop

Set up optimizer, loss function, and training procedure.

---

# STEP 7.5: Legalization Functions

Convert raw GNN predictions to legal, row-aligned placements.

**Important:** GNN produces global placement (continuous coordinates). This step performs:
- Row alignment (snap to placement rows)
- Site snapping (align to grid sites)
- Overlap removal (prevent cell overlaps)

In [8]:
def legalize_placement(predictions, chip_width_um=1000, chip_height_um=1000, 
                       row_height_um=2.0, site_width_um=0.2):
    """
    Legalize GNN placement predictions to meet physical design constraints
    (Vectorized version - handles 50K+ cells in seconds)
    
    Steps:
    1. Scale from [0,1] to real chip dimensions (microns)
    2. Snap Y-coordinates to placement rows (vectorized)
    3. Snap X-coordinates to site grid (vectorized)
    4. Remove overlaps per-row using sorting (O(n log n) instead of O(n²))
    
    Args:
        predictions: Coordinates from GNN (already scaled to microns or [0,1])
        chip_width_um: Chip width in microns
        chip_height_um: Chip height in microns
        row_height_um: Standard cell row height
        site_width_um: Placement site width
    
    Returns:
        legalized_coords: Physical coordinates aligned to rows and sites
    """
    import time
    t0 = time.time()
    n_cells = len(predictions)
    print(f"   Legalizing {n_cells:,} cells...")
    
    # Step 1: Scale to real dimensions
    scaled_coords = predictions.copy()
    scaled_coords[:, 0] = np.clip(scaled_coords[:, 0], 0, chip_width_um)
    scaled_coords[:, 1] = np.clip(scaled_coords[:, 1], 0, chip_height_um)
    
    # Step 2: Snap to rows (vectorized - no loop)
    num_rows = int(chip_height_um / row_height_um)
    row_indices = np.clip((scaled_coords[:, 1] / row_height_um).astype(int), 0, num_rows - 1)
    scaled_coords[:, 1] = row_indices * row_height_um + row_height_um / 2
    
    # Step 3: Snap to sites (vectorized - no loop)
    num_sites = int(chip_width_um / site_width_um)
    site_indices = np.clip((scaled_coords[:, 0] / site_width_um).astype(int), 0, num_sites - 1)
    scaled_coords[:, 0] = site_indices * site_width_um
    
    print(f"   Row/site snapping done in {time.time()-t0:.2f}s")
    
    # Step 4: Overlap removal PER ROW using sorting (O(n log n) total)
    legalized = scaled_coords.copy()
    cell_width = site_width_um * 5  # Assume cells are ~5 sites wide
    
    unique_rows = np.unique(legalized[:, 1])
    overlaps_fixed = 0
    
    for row_y in unique_rows:
        # Get indices of cells in this row
        row_mask = legalized[:, 1] == row_y
        row_indices_arr = np.where(row_mask)[0]
        
        if len(row_indices_arr) <= 1:
            continue
        
        # Sort cells in this row by X coordinate
        x_vals = legalized[row_indices_arr, 0]
        sort_order = np.argsort(x_vals)
        sorted_indices = row_indices_arr[sort_order]
        
        # Single pass: push overlapping cells to the right
        for k in range(1, len(sorted_indices)):
            prev_idx = sorted_indices[k - 1]
            curr_idx = sorted_indices[k]
            gap = legalized[curr_idx, 0] - legalized[prev_idx, 0]
            if gap < cell_width:
                legalized[curr_idx, 0] = legalized[prev_idx, 0] + cell_width
                overlaps_fixed += 1
        
        # Clip to chip boundary
        legalized[sorted_indices, 0] = np.clip(legalized[sorted_indices, 0], 0, chip_width_um)
    
    # Final site snap after overlap removal (vectorized)
    site_indices = np.clip((legalized[:, 0] / site_width_um).astype(int), 0, num_sites - 1)
    legalized[:, 0] = site_indices * site_width_um
    
    elapsed = time.time() - t0
    print(f"   Legalization complete in {elapsed:.2f}s ({overlaps_fixed:,} overlaps fixed)")
    
    return legalized


def compare_global_vs_legal(predictions, legalized, sample_name="sample"):
    """
    Visualize difference between global placement and legalized placement
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Subsample for plotting if too many points (faster rendering)
    n = len(predictions)
    if n > 10000:
        idx = np.random.choice(n, 10000, replace=False)
        pred_plot = predictions[idx]
        legal_plot = legalized[idx]
        plot_label = f" (10K of {n:,} shown)"
    else:
        pred_plot = predictions
        legal_plot = legalized
        plot_label = ""
    
    # Global placement (raw GNN output)
    ax1 = axes[0]
    ax1.scatter(pred_plot[:, 0], pred_plot[:, 1], c='red', s=5, alpha=0.6, label='Global Placement')
    ax1.set_xlabel('X (µm)', fontsize=12)
    ax1.set_ylabel('Y (µm)', fontsize=12)
    ax1.set_title(f'Global Placement (Raw GNN){plot_label}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Legalized placement
    ax2 = axes[1]
    ax2.scatter(legal_plot[:, 0], legal_plot[:, 1], c='green', s=5, alpha=0.6, label='Legalized Placement')
    ax2.set_xlabel('X (µm)', fontsize=12)
    ax2.set_ylabel('Y (µm)', fontsize=12)
    ax2.set_title(f'Legalized Placement (Row-Aligned){plot_label}', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(f'{sample_name}_global_vs_legal.png', dpi=100)
    plt.show()
    
    # Calculate displacement
    displacement = np.linalg.norm(legalized - predictions, axis=1)
    print(f"\nLegalization Impact:")
    print(f"   Average displacement: {np.mean(displacement):.2f} µm")
    print(f"   Max displacement: {np.max(displacement):.2f} µm")
    print(f"   Cells moved: {np.sum(displacement > 0.1)}/{len(displacement)}")
    
    # Check row alignment
    unique_rows = len(np.unique(legalized[:, 1]))
    print(f"   Cells placed in {unique_rows} rows")
    
    return displacement


print("Legalization functions defined!")
print("\nNote: GNN performs GLOBAL PLACEMENT only.")
print("Legalization converts continuous predictions to legal placements.")

Legalization functions defined!

Note: GNN performs GLOBAL PLACEMENT only.
Legalization converts continuous predictions to legal placements.


In [9]:
# ================================================================
# TRAINING CONFIGURATION (v2 - Industry-Aware Loss)
# ================================================================
BATCH_SIZE = 1       # Process one design at a time (large graphs)
LEARNING_RATE = 0.001
NUM_EPOCHS = 100     # Increased from 20 — needed for 52K-cell graphs

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# Learning Rate Scheduler - cosine decay (smooth warmdown)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# ================================================================
# CUSTOM LOSS FUNCTION — fixes center collapse + overlap
# ================================================================
def vlsi_placement_loss(pred, target, data):
    """
    Industry-aware VLSI placement loss with 4 components:
    
    1. Weighted MSE     — macros and high-connectivity cells matter more
    2. Spread Loss      — prevent center collapse (match std of predictions to targets)
    3. Range Loss       — ensure predictions fill the chip area
    4. Repulsion Loss   — penalize cells placed at exactly the same location
    
    Why standard MSE fails:
    - All cells weighted equally → model ignores 3 macros among 52K cells
    - MSE minimum = predicting the mean → everything collapses to center
    - No spatial awareness → overlapping cells not penalized
    """
    device = pred.device
    is_macro = data.is_macro.to(device)
    conn_importance = data.x[:, 15].to(device)  # Feature [15] = connectivity importance
    N = pred.shape[0]
    
    # ========================================
    # 1. WEIGHTED MSE LOSS
    # ========================================
    # Base weight: 1.0 for all cells
    cell_weights = torch.ones(N, device=device)
    # Macros (PLL, SRAM): 10x weight — they anchor the design
    cell_weights = cell_weights + 9.0 * is_macro
    # High-connectivity cells: up to 3x weight — critical for wirelength
    cell_weights = cell_weights + 2.0 * conn_importance
    
    mse_per_cell = ((pred - target) ** 2).mean(dim=1)  # [N]
    weighted_mse = (mse_per_cell * cell_weights).mean()
    
    # ========================================
    # 2. SPREAD LOSS (anti-collapse)
    # ========================================
    # Force predictions to have same spread as targets
    pred_std = pred.std(dim=0)      # [2] — std in X and Y
    target_std = target.std(dim=0)  # [2]
    spread_loss = ((pred_std - target_std) ** 2).sum()
    
    # ========================================
    # 3. RANGE LOSS (fill the chip)
    # ========================================
    pred_range = pred.max(dim=0)[0] - pred.min(dim=0)[0]    # [2]
    target_range = target.max(dim=0)[0] - target.min(dim=0)[0]  # [2]
    range_loss = ((pred_range - target_range) ** 2).sum()
    
    # ========================================
    # 4. REPULSION LOSS (prevent overlap)
    # ========================================
    # Subsample for efficiency (52K cells = 2.7B pairs → too expensive)
    n_sub = min(1000, N)
    idx = torch.randperm(N, device=device)[:n_sub]
    pred_sub = pred[idx]  # [n_sub, 2]
    
    # Pairwise distances for subsample
    diff = pred_sub.unsqueeze(0) - pred_sub.unsqueeze(1)     # [n_sub, n_sub, 2]
    dist = torch.sqrt((diff ** 2).sum(dim=2) + 1e-8)         # [n_sub, n_sub]
    
    # Penalize cells closer than min_dist (avg spacing ≈ 0.004 for 52K cells)
    min_dist = 0.003
    repulsion = torch.relu(min_dist - dist)
    # Zero out diagonal (cell distance to itself)
    mask = 1.0 - torch.eye(n_sub, device=device)
    repulsion_loss = (repulsion * mask).sum() / (n_sub * (n_sub - 1) + 1e-8)
    
    # ========================================
    # COMBINED LOSS
    # ========================================
    total = (
        1.0 * weighted_mse +    # Primary objective
        0.5 * spread_loss +      # Prevent center collapse
        0.3 * range_loss +       # Fill chip area
        0.01 * repulsion_loss    # Prevent cell stacking
    )
    
    components = {
        'mse': weighted_mse.item(),
        'spread': spread_loss.item(),
        'range': range_loss.item(),
        'repulsion': repulsion_loss.item(),
    }
    
    return total, components

print("Training Configuration (v2 - Industry-Aware):")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE} → 1e-6 (cosine decay)")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Weight decay: 1e-5")
print(f"\nLoss Function Components:")
print(f"   1. Weighted MSE (1.0x) — macros 10x, high-connectivity 3x")
print(f"   2. Spread loss (0.5x)  — match prediction std to target std")
print(f"   3. Range loss (0.3x)   — fill chip area, prevent clustering")
print(f"   4. Repulsion (0.01x)   — prevent cell stacking (subsampled)")
print(f"\nOptimizer: Adam with CosineAnnealingLR scheduler")
print("Training setup complete!")

Training Configuration (v2 - Industry-Aware):
   Batch size: 1
   Learning rate: 0.001 → 1e-6 (cosine decay)
   Epochs: 100
   Weight decay: 1e-5

Loss Function Components:
   1. Weighted MSE (1.0x) — macros 10x, high-connectivity 3x
   2. Spread loss (0.5x)  — match prediction std to target std
   3. Range loss (0.3x)   — fill chip area, prevent clustering
   4. Repulsion (0.01x)   — prevent cell stacking (subsampled)

Optimizer: Adam with CosineAnnealingLR scheduler
Training setup complete!


---

# STEP 9: Train the Model

Train the GNN on CircuitNet data. This may take 15-30 minutes depending on your GPU.

In [ ]:
# ================================================================
# TRAINING LOOP (v2 - Industry-Aware with Anti-Collapse Loss)
# ================================================================
train_losses = []
test_losses = []
loss_components_history = {'mse': [], 'spread': [], 'range': [], 'repulsion': []}
start_epoch = 0

# Check if model exists and load it
model_save_path = model_save

if os.path.exists(model_save_path):
    print("=" * 80)
    print("CHECKING EXISTING MODEL")
    print("=" * 80)
    
    checkpoint = torch.load(model_save_path)
    saved_config = checkpoint.get('model_config', {})
    saved_input_dim = saved_config.get('input_dim', 10)
    saved_version = saved_config.get('version', 'v1')
    current_input_dim = model.input_dim
    
    if saved_input_dim != current_input_dim or saved_version != 'v2':
        print(f"   Model architecture changed (v1 → v2 with anti-collapse loss)")
        print(f"   Training from scratch with new architecture.")
        print("=" * 80)
        print()
    else:
        try:
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch']
            train_losses = checkpoint.get('train_losses', [])
            test_losses = checkpoint.get('test_losses', [])
            loss_components_history = checkpoint.get('loss_components', 
                {'mse': [], 'spread': [], 'range': [], 'repulsion': []})
            
            print(f"   Model loaded from: {model_save_path}")
            print(f"   Previous epochs: {start_epoch}")
            if train_losses:
                print(f"   Previous train loss: {train_losses[-1]:.6f}")
            print(f"   Resuming training from epoch {start_epoch + 1}")
            print("=" * 80)
            print()
        except Exception as e:
            print(f"   Failed to load weights: {e}")
            print(f"   Training from scratch.")
            start_epoch = 0
            train_losses = []
            test_losses = []
            print("=" * 80)
            print()
else:
    print("=" * 80)
    print("STARTING TRAINING FROM SCRATCH (v2 Architecture)")
    print("=" * 80)
    print()

print("Starting Training with Industry-Aware Loss...\n")
print("=" * 80)
print(f"{'Epoch':>6} | {'Train':>10} | {'Test':>10} | {'MSE':>8} | {'Spread':>8} | {'Range':>8} | {'Repuls':>8} | {'LR':>10} | {'Time':>6}")
print("-" * 95)

start_time = time.time()
best_test_loss = float('inf')

for epoch in range(start_epoch, start_epoch + NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    batch_count = 0
    epoch_components = {'mse': 0, 'spread': 0, 'range': 0, 'repulsion': 0}
    
    # Training
    for data in cn_train:
        data = data.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        pred = model(data)
        
        # Industry-aware loss (weighted MSE + spread + range + repulsion)
        loss, components = vlsi_placement_loss(pred, data.y, data)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping (prevents exploding gradients with custom loss)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
        for k in epoch_components:
            epoch_components[k] += components[k]
    
    avg_train_loss = epoch_loss / batch_count
    train_losses.append(avg_train_loss)
    for k in epoch_components:
        epoch_components[k] /= batch_count
        loss_components_history[k].append(epoch_components[k])
    
    # Step the learning rate scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Evaluation
    model.eval()
    test_loss = 0
    test_count = 0
    
    with torch.no_grad():
        for data in cn_test:
            data = data.to(device)
            pred = model(data)
            loss, _ = vlsi_placement_loss(pred, data.y, data)
            test_loss += loss.item()
            test_count += 1
    
    avg_test_loss = test_loss / test_count
    test_losses.append(avg_test_loss)
    
    # Track best model
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        best_marker = " ★"
    else:
        best_marker = ""
    
    # Print progress every 5 epochs or first/last
    elapsed = time.time() - start_time
    if (epoch + 1) % 5 == 0 or epoch == start_epoch or epoch == start_epoch + NUM_EPOCHS - 1:
        print(f"{epoch+1:>6} | {avg_train_loss:>10.6f} | {avg_test_loss:>10.6f} | "
              f"{epoch_components['mse']:>8.5f} | {epoch_components['spread']:>8.5f} | "
              f"{epoch_components['range']:>8.5f} | {epoch_components['repulsion']:>8.5f} | "
              f"{current_lr:>10.2e} | {elapsed:>5.0f}s{best_marker}")

print("=" * 80)
print(f"\nTraining complete! Total time: {elapsed/60:.1f} minutes")
print(f"   Final train loss: {train_losses[-1]:.6f}")
print(f"   Final test loss:  {test_losses[-1]:.6f}")
print(f"   Best test loss:   {best_test_loss:.6f}")
print(f"   Total epochs:     {start_epoch + NUM_EPOCHS}")
print(f"\nLoss components (final epoch):")
print(f"   Weighted MSE:  {epoch_components['mse']:.6f}")
print(f"   Spread loss:   {epoch_components['spread']:.6f}")
print(f"   Range loss:    {epoch_components['range']:.6f}")
print(f"   Repulsion:     {epoch_components['repulsion']:.6f}")

---

# STEP 10: Visualize Training Progress

Plot the training and test loss curves.

In [ ]:
# ================================================================
# TRAINING CURVES — Total Loss + Individual Components
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (1) Total Loss
ax = axes[0, 0]
ax.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Train Loss', linewidth=2)
ax.plot(range(1, len(test_losses) + 1), test_losses, 'r-', label='Test Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Total Loss', fontsize=12)
ax.set_title('Total Training Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# (2) Total Loss (log scale)
ax = axes[0, 1]
ax.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Train Loss', linewidth=2)
ax.plot(range(1, len(test_losses) + 1), test_losses, 'r-', label='Test Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Total Loss (log)', fontsize=12)
ax.set_title('Total Loss (Log Scale)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# (3) Loss Components
ax = axes[1, 0]
if loss_components_history and loss_components_history.get('mse'):
    epochs_range = range(1, len(loss_components_history['mse']) + 1)
    ax.plot(epochs_range, loss_components_history['mse'], 'b-', label='Weighted MSE', linewidth=2)
    ax.plot(epochs_range, loss_components_history['spread'], 'g-', label='Spread Loss', linewidth=2)
    ax.plot(epochs_range, loss_components_history['range'], 'orange', label='Range Loss', linewidth=2)
    ax.plot(epochs_range, loss_components_history['repulsion'], 'r-', label='Repulsion', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss Value', fontsize=12)
ax.set_title('Individual Loss Components', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# (4) Components (log scale)
ax = axes[1, 1]
if loss_components_history and loss_components_history.get('mse'):
    epochs_range = range(1, len(loss_components_history['mse']) + 1)
    ax.plot(epochs_range, loss_components_history['mse'], 'b-', label='Weighted MSE', linewidth=2)
    ax.plot(epochs_range, loss_components_history['spread'], 'g-', label='Spread Loss', linewidth=2)
    ax.plot(epochs_range, loss_components_history['range'], 'orange', label='Range Loss', linewidth=2)
    ax.plot(epochs_range, loss_components_history['repulsion'], 'r-', label='Repulsion', linewidth=2)
    ax.set_yscale('log')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss Value (log)', fontsize=12)
ax.set_title('Loss Components (Log Scale)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Training curves plotted!")
print(f"   Total epochs: {len(train_losses)}")
if train_losses:
    print(f"   Final train loss: {train_losses[-1]:.6f}")
    print(f"   Final test loss:  {test_losses[-1]:.6f}")

---

# STEP 11: Test Model Predictions

Run the trained model on a test sample and visualize results.

In [ ]:
import time
step11_start = time.time()

# Test on a sample
model.eval()
test_sample = cn_test[0].to(device)

with torch.no_grad():
    pred_coords = model(test_sample).cpu().numpy()

actual_coords = test_sample.y.cpu().numpy()
macro_mask = test_sample.is_macro.cpu().numpy() > 0.5
std_mask = ~macro_mask

# IMPORTANT: Perform legalization on predictions
print("=" * 80)
print("GLOBAL PLACEMENT → LEGALIZATION")
print("=" * 80)

# Scale predictions to microns for legalization
pred_microns = pred_coords * np.array([1000, 1000])
legalized_coords = legalize_placement(pred_microns, chip_width_um=1000, chip_height_um=1000)

# Visualize global vs legal
compare_global_vs_legal(pred_microns, legalized_coords, sample_name=test_sample.sample_name)

# Calculate error
error = np.sqrt(np.sum((pred_coords - actual_coords) ** 2, axis=1))
macro_error = error[macro_mask] if macro_mask.any() else np.array([0])
std_error = error[std_mask]

print(f"\nTest Sample Results (v2 - Anti-Collapse):\n")
print(f"Sample: {test_sample.sample_name}")
print(f"Cells: {test_sample.num_cells:,} ({macro_mask.sum()} macros, {std_mask.sum():,} std cells)")
print(f"\n{'Metric':<30} {'All Cells':>12} {'Macros':>12} {'Std Cells':>12}")
print("-" * 68)
print(f"{'Mean error':<30} {np.mean(error):>12.6f} {np.mean(macro_error):>12.6f} {np.mean(std_error):>12.6f}")
print(f"{'Max error':<30} {np.max(error):>12.6f} {np.max(macro_error):>12.6f} {np.max(std_error):>12.6f}")
print(f"{'Std error':<30} {np.std(error):>12.6f} {np.std(macro_error):>12.6f} {np.std(std_error):>12.6f}")

# Distribution check (anti-collapse verification)
pred_std_x, pred_std_y = np.std(pred_coords[:, 0]), np.std(pred_coords[:, 1])
act_std_x, act_std_y = np.std(actual_coords[:, 0]), np.std(actual_coords[:, 1])
print(f"\nSpread Verification (anti-collapse):")
print(f"   Predicted std:  X={pred_std_x:.4f}, Y={pred_std_y:.4f}")
print(f"   Actual std:     X={act_std_x:.4f}, Y={act_std_y:.4f}")
print(f"   Spread ratio:   X={pred_std_x/act_std_x:.2f}x, Y={pred_std_y/act_std_y:.2f}x")
collapse_check = "✓ No collapse" if pred_std_x/act_std_x > 0.5 and pred_std_y/act_std_y > 0.5 else "⚠ Possible collapse"
print(f"   Status: {collapse_check}")

# Subsample for visualization
n_total = len(actual_coords)
max_plot_points = 10000
if n_total > max_plot_points:
    plot_idx = np.random.choice(n_total, max_plot_points, replace=False)
    actual_plot = actual_coords[plot_idx]
    pred_plot = pred_coords[plot_idx]
    legal_plot = legalized_coords[plot_idx] / np.array([1000, 1000])
    error_plot = error[plot_idx]
    macro_plot = macro_mask[plot_idx]
    plot_note = f" (showing {max_plot_points:,} of {n_total:,})"
else:
    actual_plot = actual_coords
    pred_plot = pred_coords
    legal_plot = legalized_coords / np.array([1000, 1000])
    error_plot = error
    macro_plot = macro_mask
    plot_note = ""

fig, axes = plt.subplots(2, 3, figsize=(24, 14))

# Row 1: Full view
# (1) Ground Truth
axes[0, 0].scatter(actual_plot[:, 0], actual_plot[:, 1], c='blue', s=1, alpha=0.3)
if macro_mask.any():
    axes[0, 0].scatter(actual_coords[macro_mask, 0], actual_coords[macro_mask, 1],
                       c='red', s=100, marker='s', edgecolors='black', linewidth=1.5, label='Macros', zorder=5)
axes[0, 0].set_title(f'Ground Truth{plot_note}', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('X (normalized)')
axes[0, 0].set_ylabel('Y (normalized)')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# (2) GNN Predicted
axes[0, 1].scatter(pred_plot[:, 0], pred_plot[:, 1], c='forestgreen', s=1, alpha=0.3)
if macro_mask.any():
    axes[0, 1].scatter(pred_coords[macro_mask, 0], pred_coords[macro_mask, 1],
                       c='red', s=100, marker='s', edgecolors='black', linewidth=1.5, label='Macros', zorder=5)
axes[0, 1].set_title(f'GNN Predicted (v2){plot_note}', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('X (normalized)')
axes[0, 1].set_ylabel('Y (normalized)')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# (3) Legalized
axes[0, 2].scatter(legal_plot[:, 0], legal_plot[:, 1], c='purple', s=1, alpha=0.3)
if macro_mask.any():
    macro_legal = legalized_coords[macro_mask] / np.array([1000, 1000])
    axes[0, 2].scatter(macro_legal[:, 0], macro_legal[:, 1],
                       c='red', s=100, marker='s', edgecolors='black', linewidth=1.5, label='Macros', zorder=5)
axes[0, 2].set_title(f'Legalized{plot_note}', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('X (normalized)')
axes[0, 2].set_ylabel('Y (normalized)')
axes[0, 2].legend(fontsize=9)
axes[0, 2].grid(True, alpha=0.3)

# Row 2: Analysis
# (4) Error Heatmap
scatter = axes[1, 0].scatter(actual_plot[:, 0], actual_plot[:, 1],
                             c=error_plot, cmap='hot', s=1, alpha=0.7)
axes[1, 0].set_title(f'Error Heatmap{plot_note}', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('X (normalized)')
axes[1, 0].set_ylabel('Y (normalized)')
axes[1, 0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 0], label='Error')

# (5) Predicted vs Actual overlay
axes[1, 1].scatter(actual_plot[:, 0], actual_plot[:, 1], c='blue', s=1, alpha=0.2, label='Actual')
axes[1, 1].scatter(pred_plot[:, 0], pred_plot[:, 1], c='red', s=1, alpha=0.2, label='Predicted')
if macro_mask.any():
    for mi in np.where(macro_mask)[0]:
        axes[1, 1].annotate('', xy=(pred_coords[mi, 0], pred_coords[mi, 1]),
                            xytext=(actual_coords[mi, 0], actual_coords[mi, 1]),
                            arrowprops=dict(arrowstyle='->', color='lime', lw=2))
axes[1, 1].set_title(f'Overlay (blue=actual, red=pred){plot_note}', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('X (normalized)')
axes[1, 1].set_ylabel('Y (normalized)')
axes[1, 1].legend(fontsize=9, markerscale=5)
axes[1, 1].grid(True, alpha=0.3)

# (6) Displacement vectors for high-connectivity cells
conn_imp = test_sample.x[:, 15].cpu().numpy()
top_k = min(500, n_total)
top_idx = np.argsort(conn_imp)[-top_k:]
axes[1, 2].scatter(actual_coords[top_idx, 0], actual_coords[top_idx, 1], c='blue', s=5, alpha=0.5, label='Actual')
axes[1, 2].scatter(pred_coords[top_idx, 0], pred_coords[top_idx, 1], c='red', s=5, alpha=0.5, label='Predicted')
for i in top_idx[::10]:  # Draw arrows for every 10th
    axes[1, 2].annotate('', xy=(pred_coords[i, 0], pred_coords[i, 1]),
                        xytext=(actual_coords[i, 0], actual_coords[i, 1]),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=0.5, alpha=0.5))
axes[1, 2].set_title(f'Top-{top_k} Connected Cells Displacement', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('X (normalized)')
axes[1, 2].set_ylabel('Y (normalized)')
axes[1, 2].legend(fontsize=9)
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

step11_elapsed = time.time() - step11_start
print(f"\nVisualization complete! (Step 11 took {step11_elapsed:.1f}s)")
print("NOTE: Use legalized_coords for DEF export and EDA tool integration.")

---

# Understanding the Placement Flow

## The Complete VLSI Placement Pipeline:

### 1. **Global Placement (GNN Model)**
- Input: Circuit netlist as graph
- Output: Continuous (x, y) coordinates in [0, 1] range
- Goal: Minimize wirelength, spread cells across chip
- **No physical constraints** - cells can overlap, float between rows

### 2. **Legalization (Post-Processing)**
- Input: Global placement coordinates
- Output: Legal, row-aligned coordinates
- Operations:
  - Scale to real chip dimensions (microns)
  - **Snap Y to rows** (remove vertical jitter)
  - **Snap X to sites** (align to placement grid)
  - **Remove overlaps** (cells cannot occupy same space)

### 3. **Detailed Placement (Not in this notebook)**
- Fine-tune positions within rows
- Optimize for timing, power, routability
- Usually done by commercial tools (Innovus, ICC)

---

## Why This Matters:

- **Raw GNN outputs are NOT legal placements** - they're global placement hints
- **DEF files require legalized coordinates** - EDA tools expect row-aligned, non-overlapping cells
- **Industry separation:** ML does global placement, algorithms do legalization
- **This notebook shows both:** Global placement quality + legalized output

---

In [ ]:
# Save model (v2 - with loss components history)
model_save_path = model_save

# Calculate total epochs
total_epochs = start_epoch + NUM_EPOCHS

torch.save({
    'epoch': total_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'test_losses': test_losses,
    'loss_components': loss_components_history,
    'model_config': {
        'input_dim': 16,
        'hidden_dim': 128,
        'output_dim': 2,
        'num_layers': 4,
        'heads': 4,
        'version': 'v2'  # Track architecture version
    }
}, model_save_path)

print(f"Model saved to: {model_save_path}")
print(f"   Architecture: v2 (anti-collapse, residual, no sigmoid)")
print(f"   Total epochs completed: {total_epochs}")
print(f"   File size: {os.path.getsize(model_save_path) / 1e6:.2f} MB")
print(f"\nTo load later:")
print(f"   checkpoint = torch.load('{model_save_path}')")
print(f"   model.load_state_dict(checkpoint['model_state_dict'])")

---

# STEP 12.5: Load Pre-trained Model & Test on Single Sample

Load a previously saved model and test it on a single sample.

In [ ]:
# Load a pre-trained model (v2) and test on a single sample
print("Loading pre-trained model...\n")

# Path to saved model
model_load_path = model_save

# Check if model file exists
if not os.path.exists(model_load_path):
    print(f"ERROR: Model file not found at {model_load_path}")
    print("Please train the model first (run the training cell)")
else:
    # Load checkpoint
    checkpoint = torch.load(model_load_path, map_location=device)
    
    # Create a new model instance (v2 architecture)
    loaded_model = VLSIPlacementGNN(
        input_dim=16,
        hidden_dim=128,
        output_dim=2,
        num_layers=4,
        heads=4
    ).to(device)
    
    # Load saved weights
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model.eval()
    
    config = checkpoint.get('model_config', {})
    version = config.get('version', 'v1')
    
    print(f"Model loaded successfully!")
    print(f"   Architecture: {version}")
    print(f"   Trained for: {checkpoint['epoch']} epochs")
    print(f"   Final train loss: {checkpoint['train_losses'][-1]:.6f}")
    print(f"   Final test loss: {checkpoint['test_losses'][-1]:.6f}")
    
    # Test on a single sample
    print("\n" + "="*80)
    print("TESTING ON SINGLE SAMPLE")
    print("="*80)
    
    # Select a single test sample
    test_idx = min(10, len(cn_test) - 1)
    # test_sample = cn_test[test_idx].to(device)
    test_sample = cn_test[29].to(device)
    
    
    print(f"\nTest Sample Details:")
    print(f"   Sample name: {test_sample.sample_name}")
    print(f"   Design: {test_sample.design_name}")
    print(f"   Number of cells: {test_sample.num_cells:,}")
    print(f"   Number of edges: {test_sample.edge_index.shape[1]:,}")
    
    # Run inference
    with torch.no_grad():
        pred_coords = loaded_model(test_sample).cpu().numpy()
    
    actual_coords = test_sample.y.cpu().numpy()
    macro_mask = test_sample.is_macro.cpu().numpy() > 0.5
    
    # Calculate error metrics
    error = np.sqrt(np.sum((pred_coords - actual_coords) ** 2, axis=1))
    
    print(f"\nPrediction Results:")
    print(f"   Mean placement error: {np.mean(error):.6f}")
    print(f"   Max placement error: {np.max(error):.6f}")
    print(f"   Macro error: {np.mean(error[macro_mask]):.6f}" if macro_mask.any() else "   No macros")
    
    # Spread verification
    pred_std = np.std(pred_coords, axis=0)
    act_std = np.std(actual_coords, axis=0)
    print(f"\n   Spread ratio: X={pred_std[0]/act_std[0]:.2f}x, Y={pred_std[1]/act_std[1]:.2f}x")
    
    # Visualize single sample prediction
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Ground truth
    axes[0].scatter(actual_coords[:, 0], actual_coords[:, 1], c='blue', s=2, alpha=0.6)
    if macro_mask.any():
        axes[0].scatter(actual_coords[macro_mask, 0], actual_coords[macro_mask, 1],
                       c='red', s=80, marker='s', edgecolors='black', zorder=5, label='Macros')
        axes[0].legend()
    axes[0].set_title('Ground Truth Placement', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X (normalized)', fontsize=12)
    axes[0].set_ylabel('Y (normalized)', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(-0.05, 1.05)
    
    # Predicted placement
    axes[1].scatter(pred_coords[:, 0], pred_coords[:, 1], c='forestgreen', s=2, alpha=0.6)
    if macro_mask.any():
        axes[1].scatter(pred_coords[macro_mask, 0], pred_coords[macro_mask, 1],
                       c='red', s=80, marker='s', edgecolors='black', zorder=5, label='Macros')
        axes[1].legend()
    axes[1].set_title(f'Predicted Placement ({version})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('X (normalized)', fontsize=12)
    axes[1].set_ylabel('Y (normalized)', fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    
    # Error heatmap
    scatter = axes[2].scatter(actual_coords[:, 0], actual_coords[:, 1], 
                             c=error, cmap='hot', s=2, alpha=0.8, vmin=0, vmax=np.percentile(error, 95))
    axes[2].set_title('Placement Error Heatmap', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('X (normalized)', fontsize=12)
    axes[2].set_ylabel('Y (normalized)', fontsize=12)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim(-0.05, 1.05)
    axes[2].set_ylim(-0.05, 1.05)
    cbar = plt.colorbar(scatter, ax=axes[2], label='Error')
    
    plt.tight_layout()
    plt.savefig('loaded_model_test.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"\nSingle sample test complete!")
    print(f"   Visualization saved as: loaded_model_test.png")

In [ ]:
# Load a pre-trained model (v2) and test on a single sample
print("Loading pre-trained model...\n")

# Path to saved model
model_load_path = model_save

# Check if model file exists
if not os.path.exists(model_load_path):
    print(f"ERROR: Model file not found at {model_load_path}")
    print("Please train the model first (run the training cell)")
else:
    # Load checkpoint
    checkpoint = torch.load(model_load_path, map_location=device)
    
    # Create a new model instance (v2 architecture)
    loaded_model = VLSIPlacementGNN(
        input_dim=16,
        hidden_dim=128,
        output_dim=2,
        num_layers=4,
        heads=4
    ).to(device)
    
    # Load saved weights
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model.eval()
    
    config = checkpoint.get('model_config', {})
    version = config.get('version', 'v1')
    
    print(f"Model loaded successfully!")
    print(f"   Architecture: {version}")
    print(f"   Trained for: {checkpoint['epoch']} epochs")
    print(f"   Final train loss: {checkpoint['train_losses'][-1]:.6f}")
    print(f"   Final test loss: {checkpoint['test_losses'][-1]:.6f}")
    
    # Test on a single sample
    print("\n" + "="*80)
    print("TESTING ON SINGLE SAMPLE")
    print("="*80)
    
    # Select a single test sample
    test_idx = min(10, len(cn_test) - 1)
    # test_sample = cn_test[test_idx].to(device)
    test_sample = cn_test[29].to(device)
    
    
    print(f"\nTest Sample Details:")
    print(f"   Sample name: {test_sample.sample_name}")
    print(f"   Design: {test_sample.design_name}")
    print(f"   Number of cells: {test_sample.num_cells:,}")
    print(f"   Number of edges: {test_sample.edge_index.shape[1]:,}")
    
    # Run inference
    with torch.no_grad():
        pred_coords = loaded_model(test_sample).cpu().numpy()
    
    actual_coords = test_sample.y.cpu().numpy()
    macro_mask = test_sample.is_macro.cpu().numpy() > 0.5
    
    # Calculate error metrics
    error = np.sqrt(np.sum((pred_coords - actual_coords) ** 2, axis=1))
    
    print(f"\nPrediction Results:")
    print(f"   Mean placement error: {np.mean(error):.6f}")
    print(f"   Max placement error: {np.max(error):.6f}")
    print(f"   Macro error: {np.mean(error[macro_mask]):.6f}" if macro_mask.any() else "   No macros")
    
    # Spread verification
    pred_std = np.std(pred_coords, axis=0)
    act_std = np.std(actual_coords, axis=0)
    print(f"\n   Spread ratio: X={pred_std[0]/act_std[0]:.2f}x, Y={pred_std[1]/act_std[1]:.2f}x")
    
    # Visualize single sample prediction
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Ground truth
    axes[0].scatter(actual_coords[:, 0], actual_coords[:, 1], c='blue', s=2, alpha=0.6)
    if macro_mask.any():
        axes[0].scatter(actual_coords[macro_mask, 0], actual_coords[macro_mask, 1],
                       c='red', s=80, marker='s', edgecolors='black', zorder=5, label='Macros')
        axes[0].legend()
    axes[0].set_title('Ground Truth Placement', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X (normalized)', fontsize=12)
    axes[0].set_ylabel('Y (normalized)', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(-0.05, 1.05)
    
    # Predicted placement
    axes[1].scatter(pred_coords[:, 0], pred_coords[:, 1], c='forestgreen', s=2, alpha=0.6)
    if macro_mask.any():
        axes[1].scatter(pred_coords[macro_mask, 0], pred_coords[macro_mask, 1],
                       c='red', s=80, marker='s', edgecolors='black', zorder=5, label='Macros')
        axes[1].legend()
    axes[1].set_title(f'Predicted Placement ({version})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('X (normalized)', fontsize=12)
    axes[1].set_ylabel('Y (normalized)', fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    
    # Error heatmap
    scatter = axes[2].scatter(actual_coords[:, 0], actual_coords[:, 1], 
                             c=error, cmap='hot', s=2, alpha=0.8, vmin=0, vmax=np.percentile(error, 95))
    axes[2].set_title('Placement Error Heatmap', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('X (normalized)', fontsize=12)
    axes[2].set_ylabel('Y (normalized)', fontsize=12)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim(-0.05, 1.05)
    axes[2].set_ylim(-0.05, 1.05)
    cbar = plt.colorbar(scatter, ax=axes[2], label='Error')
    
    plt.tight_layout()
    plt.savefig('loaded_model_test.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"\nSingle sample test complete!")
    print(f"   Visualization saved as: loaded_model_test.png")

In [ ]:
print(len(cn_test))

---

# STEP 13: Industry-Grade Visualization

Create production-quality layout visualization with micron precision.

In [ ]:
def visualize_industry_layout(data, predictions, chip_width_microns=1000, chip_height_microns=1000, dpi=150):
    """
    Industry-grade layout visualization with micron precision
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=dpi)
    
    # Convert normalized coordinates to microns
    predicted_microns = predictions * np.array([chip_width_microns, chip_height_microns])
    actual_microns = data.y.cpu().numpy() * np.array([chip_width_microns, chip_height_microns])
    
    # Plot 1: Predicted Layout
    ax1 = axes[0]
    ax1.set_xlim(0, chip_width_microns)
    ax1.set_ylim(0, chip_height_microns)
    ax1.set_aspect('equal')
    ax1.set_facecolor('#1a1a1a')
    
    # Draw cells with different colors based on size
    for i, (x, y) in enumerate(predicted_microns):
        # Estimate cell size from node features (simplified)
        cell_width = max(5, min(50, data.x[i, 3].item() * 20)) if data.x.shape[1] > 3 else 10
        cell_height = cell_width * 0.8
        
        rect = Rectangle((x - cell_width/2, y - cell_height/2), 
                        cell_width, cell_height,
                        facecolor='cyan', edgecolor='white', 
                        alpha=0.7, linewidth=0.5)
        ax1.add_patch(rect)
    
    # Draw connections (sample)
    edge_index = data.edge_index.cpu().numpy()
    for i in range(0, min(500, edge_index.shape[1]), 5):  # Sample edges
        src, dst = edge_index[:, i]
        ax1.plot([predicted_microns[src, 0], predicted_microns[dst, 0]],
                [predicted_microns[src, 1], predicted_microns[dst, 1]],
                'yellow', alpha=0.2, linewidth=0.3)
    
    ax1.set_title('Predicted Layout (Micron Precision)', fontsize=14, color='white', pad=20)
    ax1.set_xlabel('X Position (µm)', fontsize=12, color='white')
    ax1.set_ylabel('Y Position (µm)', fontsize=12, color='white')
    ax1.tick_params(colors='white')
    ax1.grid(True, alpha=0.2, color='gray')
    
    # Plot 2: Actual Layout
    ax2 = axes[1]
    ax2.set_xlim(0, chip_width_microns)
    ax2.set_ylim(0, chip_height_microns)
    ax2.set_aspect('equal')
    ax2.set_facecolor('#1a1a1a')
    
    for i, (x, y) in enumerate(actual_microns):
        cell_width = max(5, min(50, data.x[i, 3].item() * 20)) if data.x.shape[1] > 3 else 10
        cell_height = cell_width * 0.8
        
        rect = Rectangle((x - cell_width/2, y - cell_height/2), 
                        cell_width, cell_height,
                        facecolor='lime', edgecolor='white', 
                        alpha=0.7, linewidth=0.5)
        ax2.add_patch(rect)
    
    # Draw connections
    for i in range(0, min(500, edge_index.shape[1]), 5):
        src, dst = edge_index[:, i]
        ax2.plot([actual_microns[src, 0], actual_microns[dst, 0]],
                [actual_microns[src, 1], actual_microns[dst, 1]],
                'yellow', alpha=0.2, linewidth=0.3)
    
    ax2.set_title('Actual Layout (Ground Truth)', fontsize=14, color='white', pad=20)
    ax2.set_xlabel('X Position (µm)', fontsize=12, color='white')
    ax2.set_ylabel('Y Position (µm)', fontsize=12, color='white')
    ax2.tick_params(colors='white')
    ax2.grid(True, alpha=0.2, color='gray')
    
    plt.tight_layout()
    plt.savefig('industry_layout.png', dpi=dpi, facecolor='#1a1a1a')
    plt.show()
    
    # Calculate metrics
    mse = np.mean((predicted_microns - actual_microns) ** 2)
    mae = np.mean(np.abs(predicted_microns - actual_microns))
    
    print(f"\nLayout Accuracy Metrics:")
    print(f"   Mean Squared Error: {mse:.2f} µm²")
    print(f"   Mean Absolute Error: {mae:.2f} µm")
    print(f"   Average X Error: {np.mean(np.abs(predicted_microns[:, 0] - actual_microns[:, 0])):.2f} µm")
    print(f"   Average Y Error: {np.mean(np.abs(predicted_microns[:, 1] - actual_microns[:, 1])):.2f} µm")

# Visualize with test data
print("Creating industry-grade layout visualization...")
model.eval()
with torch.no_grad():
    test_data = cn_test[10].to(device)
    test_pred = model(test_data).cpu().numpy()

visualize_industry_layout(test_data, test_pred)
print("Industry-grade visualization complete!")

---

# STEP 14: Wirelength Optimization

Calculate and optimize total wirelength using Half-Perimeter Wire Length (HPWL).

In [ ]:
def calculate_wirelength(data, predictions):
    """
    Calculate total wirelength using Half-Perimeter Wire Length (HPWL)
    """
    print("Calculating wirelength metrics...")
    
    edge_index = data.edge_index.cpu().numpy()
    
    # Group edges by nets (simplified: each edge is a 2-pin net)
    wirelengths = []
    
    for i in range(edge_index.shape[1]):
        src, dst = edge_index[:, i]
        x1, y1 = predictions[src]
        x2, y2 = predictions[dst]
        
        # Manhattan distance (L1 norm)
        manhattan = abs(x2 - x1) + abs(y2 - y1)
        
        # Euclidean distance (L2 norm)
        euclidean = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        
        wirelengths.append({
            'manhattan': manhattan,
            'euclidean': euclidean,
            'src': src,
            'dst': dst
        })
    
    # Calculate statistics
    total_manhattan = sum(w['manhattan'] for w in wirelengths)
    total_euclidean = sum(w['euclidean'] for w in wirelengths)
    avg_manhattan = np.mean([w['manhattan'] for w in wirelengths])
    avg_euclidean = np.mean([w['euclidean'] for w in wirelengths])
    
    # Visualize wirelength distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax1 = axes[0]
    manhattan_lengths = [w['manhattan'] for w in wirelengths]
    ax1.hist(manhattan_lengths, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax1.axvline(avg_manhattan, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_manhattan:.4f}')
    ax1.set_xlabel('Wire Length (Manhattan)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.set_title('Wirelength Distribution', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Cumulative distribution
    ax2 = axes[1]
    sorted_lengths = np.sort(manhattan_lengths)
    cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
    ax2.plot(sorted_lengths, cumulative, linewidth=2, color='green')
    ax2.set_xlabel('Wire Length (Manhattan)', fontsize=12)
    ax2.set_ylabel('Cumulative Probability', fontsize=12)
    ax2.set_title('Cumulative Wirelength Distribution', fontsize=14)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('wirelength_analysis.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    # Print results
    print(f"\nWirelength Analysis Results:")
    print(f"   Total wires: {len(wirelengths)}")
    print(f"   Total Manhattan wirelength: {total_manhattan:.4f}")
    print(f"   Total Euclidean wirelength: {total_euclidean:.4f}")
    print(f"   Average Manhattan length: {avg_manhattan:.4f}")
    print(f"   Average Euclidean length: {avg_euclidean:.4f}")
    print(f"   Maximum wire length: {max(manhattan_lengths):.4f}")
    print(f"   Minimum wire length: {min(manhattan_lengths):.4f}")
    print(f"   Std deviation: {np.std(manhattan_lengths):.4f}")
    
    # Find longest wires
    longest_wires = sorted(wirelengths, key=lambda x: x['manhattan'], reverse=True)[:5]
    print(f"\nTop 5 Longest Wires:")
    for idx, wire in enumerate(longest_wires, 1):
        print(f"   {idx}. Node {wire['src']} → Node {wire['dst']}: {wire['manhattan']:.4f}")
    
    return wirelengths, total_manhattan

# Calculate wirelength
model.eval()
with torch.no_grad():
    test_data = cn_test[0].to(device)
    test_pred = model(test_data).cpu().numpy()

wirelengths, total_wl = calculate_wirelength(test_data, test_pred)
print("\nWirelength optimization complete!")

---

# STEP 15: Multi-Sample Evaluation

Evaluate model performance across multiple test samples.

In [ ]:
# Evaluate on all test samples
print("Evaluating model on all test samples...\n")

all_results = []
model.eval()

with torch.no_grad():
    for idx, data in enumerate(cn_test):
        data = data.to(device)
        pred = model(data).cpu().numpy()
        actual = data.y.cpu().numpy()
        
        # Calculate metrics
        mse = np.mean((pred - actual) ** 2)
        mae = np.mean(np.abs(pred - actual))
        max_error = np.max(np.abs(pred - actual))
        
        all_results.append({
            'sample': data.sample_name,
            'num_cells': data.num_cells,
            'mse': mse,
            'mae': mae,
            'max_error': max_error
        })
        
        if (idx + 1) % 5 == 0:
            print(f"   Evaluated {idx+1}/{len(cn_test)} samples...")

print(f"\nEvaluation complete!\n")

# Aggregate statistics
avg_mse = np.mean([r['mse'] for r in all_results])
avg_mae = np.mean([r['mae'] for r in all_results])
avg_max_error = np.mean([r['max_error'] for r in all_results])

print("Overall Model Performance:")
print(f"   Average MSE: {avg_mse:.6f}")
print(f"   Average MAE: {avg_mae:.6f}")
print(f"   Average Max Error: {avg_max_error:.6f}")

# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MSE distribution
ax1 = axes[0, 0]
mse_values = [r['mse'] for r in all_results]
ax1.hist(mse_values, bins=20, color='blue', edgecolor='black', alpha=0.7)
ax1.axvline(avg_mse, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_mse:.6f}')
ax1.set_xlabel('MSE', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('MSE Distribution Across Test Samples', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# MAE distribution
ax2 = axes[0, 1]
mae_values = [r['mae'] for r in all_results]
ax2.hist(mae_values, bins=20, color='green', edgecolor='black', alpha=0.7)
ax2.axvline(avg_mae, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_mae:.6f}')
ax2.set_xlabel('MAE', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('MAE Distribution Across Test Samples', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Error vs. Circuit Size
ax3 = axes[1, 0]
sizes = [r['num_cells'] for r in all_results]
ax3.scatter(sizes, mae_values, alpha=0.6, s=50, color='purple')
ax3.set_xlabel('Number of Cells', fontsize=12)
ax3.set_ylabel('MAE', fontsize=12)
ax3.set_title('Placement Error vs. Circuit Size', fontsize=14)
ax3.grid(True, alpha=0.3)

# Per-sample performance
ax4 = axes[1, 1]
sample_indices = range(len(all_results))
ax4.plot(sample_indices, mse_values, 'o-', label='MSE', alpha=0.7)
ax4.plot(sample_indices, mae_values, 's-', label='MAE', alpha=0.7)
ax4.set_xlabel('Sample Index', fontsize=12)
ax4.set_ylabel('Error', fontsize=12)
ax4.set_title('Per-Sample Performance', fontsize=14)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('multi_sample_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

# Best and worst samples
best_sample = min(all_results, key=lambda x: x['mae'])
worst_sample = max(all_results, key=lambda x: x['mae'])

print(f"\nBest Sample:")
print(f"   Name: {best_sample['sample']}")
print(f"   Cells: {best_sample['num_cells']:,}")
print(f"   MAE: {best_sample['mae']:.6f}")

print(f"\nWorst Sample:")
print(f"   Name: {worst_sample['sample']}")
print(f"   Cells: {worst_sample['num_cells']:,}")
print(f"   MAE: {worst_sample['mae']:.6f}")

print("\nMulti-sample evaluation complete!")

---

# STEP 16: Export Results

Export placement results and statistics to files for EDA tool integration.

In [ ]:
import json
import csv
import numpy as np

# Custom JSON encoder to handle numpy types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# Create output directory
output_dir = output_dir
os.makedirs(output_dir, exist_ok=True)

print("Exporting results...\n")

# 1. Export placement predictions for a sample
test_sample = cn_test[0]
model.eval()
with torch.no_grad():
    test_sample_device = test_sample.to(device)
    predictions = model(test_sample_device).cpu().numpy()

# Export to JSON
placement_data = {
    'design_name': test_sample.design_name,
    'sample_name': test_sample.sample_name,
    'num_cells': int(test_sample.num_cells),
    'predictions': []
}

for i in range(len(predictions)):
    placement_data['predictions'].append({
        'cell_id': i,
        'x': float(predictions[i, 0]),
        'y': float(predictions[i, 1]),
        'x_actual': float(test_sample.y[i, 0]),
        'y_actual': float(test_sample.y[i, 1])
    })

json_path = os.path.join(output_dir, f"{test_sample.sample_name}_placement.json")
with open(json_path, 'w') as f:
    json.dump(placement_data, f, indent=2, cls=NumpyEncoder)

print(f"[OK] Exported placement to: {json_path}")

# 2. Export DEF-like format (simplified)
def_path = os.path.join(output_dir, f"{test_sample.sample_name}_placement.def")
with open(def_path, 'w') as f:
    f.write(f"DESIGN {test_sample.design_name} ;\n")
    f.write(f"COMPONENTS {test_sample.num_cells} ;\n")
    
    for i in range(len(predictions)):
        # Convert normalized coords to micron (assuming 1000x1000 um chip)
        x_um = int(predictions[i, 0] * 1000)
        y_um = int(predictions[i, 1] * 1000)
        f.write(f"  - CELL_{i} + PLACED ( {x_um} {y_um} ) N ;\n")
    
    f.write("END COMPONENTS\n")

print(f"[OK] Exported DEF format to: {def_path}")

# 3. Export evaluation metrics
metrics_data = {
    'model_info': {
        'input_dim': 16,
        'hidden_dim': 128,
        'output_dim': 2,
        'num_layers': 4,
        'heads': 4,
        'total_parameters': sum(p.numel() for p in model.parameters())
    },
    'training_info': {
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'final_train_loss': float(train_losses[-1]),
        'final_test_loss': float(test_losses[-1]),
        'num_train_samples': len(cn_train),
        'num_test_samples': len(cn_test)
    },
    'test_results': all_results
}

metrics_path = os.path.join(output_dir, "model_metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(metrics_data, f, indent=2, cls=NumpyEncoder)

print(f"[OK] Exported metrics to: {metrics_path}")

# 4. Export CSV summary
csv_path = os.path.join(output_dir, "test_results_summary.csv")
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sample', 'num_cells', 'mse', 'mae', 'max_error'])
    writer.writeheader()
    writer.writerows(all_results)

print(f"[OK] Exported CSV summary to: {csv_path}")

# 5. Export training history
history_path = os.path.join(output_dir, "training_history.csv")
with open(history_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'test_loss'])
    for epoch in range(NUM_EPOCHS):
        writer.writerow([epoch + 1, train_losses[epoch], test_losses[epoch]])

print(f"[OK] Exported training history to: {history_path}")

print(f"\nAll results exported to: {output_dir}")
print("\nExport complete!")

---

# STEP 17: Run GNN Placement Inference via GUI

Browse and select your **standard VLSI design files** — the GNN predicts cell placement from scratch:
- **Verilog Netlist** (`.v`) — cell instances, nets, connectivity *(required)*
- **LEF Library** (`.lef`) — cell dimensions & pin definitions *(required)*
- **Model checkpoint** (`.pth`) — trained GNN weights *(required)*
- **SDC file** (`.sdc`) — timing constraints *(optional)*
- **Floorplan DEF** (`.def`) — die area & IO pad positions only, NOT cell placement *(optional)*

The cell parses these files, converts them into the 16-feature graph, and the **GNN predicts placement coordinates from scratch**.

In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import re
import json

# ═══════════════════════════════════════════════════════════════════
# PARSERS: Standard VLSI files → GNN-compatible format
# ═══════════════════════════════════════════════════════════════════

def parse_verilog_netlist(verilog_path):
    """
    Parse a gate-level Verilog netlist (.v) to extract cell instances,
    net connections, and pin-level connectivity.
    
    Handles instantiations like:
      NAND2X1 u123 ( .A(n45), .B(n67), .Y(n89) );
    
    Returns:
      node_attr: [cell_names, cell_types]
      pin_attr: [nets, cells, pins]  (nets as list of pin lists)
      cell_names: list of instance names
      cell_types: list of cell types
    """
    with open(verilog_path, 'r') as f:
        content = f.read()
    
    # Remove comments
    content = re.sub(r'//.*', '', content)
    content = re.sub(r'/\*.*?\*/', '', content, flags=re.DOTALL)
    
    # Extract module name
    module_match = re.search(r'module\s+(\w+)', content)
    module_name = module_match.group(1) if module_match else "design"
    
    # Extract cell instances (gate instantiations)
    # Pattern: CellType InstName ( .pin(net), .pin(net), ... );
    inst_pattern = r'(\w+)\s+(\w+)\s*\((.*?)\)\s*;'
    instances = []
    
    for match in re.finditer(inst_pattern, content, re.DOTALL):
        cell_type = match.group(1)
        inst_name = match.group(2)
        pins_str = match.group(3)
        
        # Skip module declaration, input/output wires
        if cell_type.lower() in ['module', 'input', 'output', 'wire', 'assign']:
            continue
        
        # Parse pin connections: .A(net1), .B(net2)
        pin_pattern = r'\.(\w+)\s*\(\s*(\w+)\s*\)'
        pins = {}
        for pm in re.finditer(pin_pattern, pins_str):
            pin_name = pm.group(1)
            net_name = pm.group(2)
            pins[pin_name] = net_name
        
        instances.append({
            'name': inst_name,
            'type': cell_type,
            'pins': pins
        })
    
    if not instances:
        raise ValueError(f"No cell instances found in {verilog_path}. Check Verilog syntax.")
    
    cell_names = [inst['name'] for inst in instances]
    cell_types = [inst['type'] for inst in instances]
    
    # Build net-to-pins mapping for connectivity
    net_to_pins = {}
    for inst in instances:
        for pin_name, net_name in inst['pins'].items():
            if net_name not in net_to_pins:
                net_to_pins[net_name] = []
            net_to_pins[net_name].append((inst['name'], pin_name))
    
    # Convert to pin_attr format matching CircuitNet expectations
    # pin_attr[0] = pin name (string)
    # pin_attr[1] = net index (int) - which net this pin belongs to
    # pin_attr[2] = node index (int) - which cell this pin belongs to
    
    # Create mappings
    nets = sorted(net_to_pins.keys())  # List of net names
    net_to_idx = {net: i for i, net in enumerate(nets)}  # net name -> net index
    cell_to_idx = {name: i for i, name in enumerate(cell_names)}  # cell name -> cell index
    
    pin_attr_data = [[], [], []]
    
    for net_name, pins_on_net in net_to_pins.items():
        net_idx = net_to_idx[net_name]
        for cell_name, pin_name in pins_on_net:
            cell_idx = cell_to_idx[cell_name]
            pin_attr_data[0].append(pin_name)      # pin name (string)
            pin_attr_data[1].append(net_idx)       # net index (int)
            pin_attr_data[2].append(cell_idx)      # cell/node index (int)
    
    pin_attr = np.array(pin_attr_data, dtype=object)
    node_attr = np.array([cell_names, cell_types], dtype=object)
    
    print(f"   [Verilog] {len(cell_names)} cells, {len(cell_types)} types, {len(nets)} nets")
    return node_attr, pin_attr, cell_names, cell_types


def parse_lef_file(lef_path):
    """
    Parse LEF file to extract cell dimensions (width, height).
    Returns: cell_type -> {width, height, class, pins}
    """
    with open(lef_path, 'r') as f:
        content = f.read()
    
    cell_lib = {}
    
    # Match MACRO blocks
    macro_pattern = r'MACRO\s+(\w+)\s+(.*?)END\s+\1'
    for match in re.finditer(macro_pattern, content, re.DOTALL | re.IGNORECASE):
        cell_name = match.group(1)
        cell_block = match.group(2)
        
        # Extract SIZE dimensions
        size_match = re.search(r'SIZE\s+([\d.]+)\s+BY\s+([\d.]+)', cell_block, re.IGNORECASE)
        if size_match:
            width = float(size_match.group(1))
            height = float(size_match.group(2))
        else:
            width, height = 1.0, 1.0
        
        # Extract CLASS
        class_match = re.search(r'CLASS\s+(\w+)', cell_block, re.IGNORECASE)
        cell_class = class_match.group(1) if class_match else "CORE"
        
        # Count pins
        pin_count = len(re.findall(r'PIN\s+\w+', cell_block, re.IGNORECASE))
        
        cell_lib[cell_name] = {
            'width': width, 'height': height,
            'class': cell_class, 'pins': pin_count
        }
    
    print(f"   [LEF] {len(cell_lib)} cell definitions extracted")
    return cell_lib


def parse_floorplan_def(def_path):
    """
    Parse DEF floorplan file to extract die area and fixed IO pads ONLY.
    Does NOT extract cell placements (GNN predicts those).
    """
    with open(def_path, 'r') as f:
        content = f.read()
    
    # Extract DIEAREA
    die_area = None
    die_match = re.search(r'DIEAREA\s*\(\s*(-?\d+)\s+(-?\d+)\s*\)\s*\(\s*(-?\d+)\s+(-?\d+)\s*\)', content)
    if die_match:
        die_area = (
            int(die_match.group(1)), int(die_match.group(2)),
            int(die_match.group(3)), int(die_match.group(4))
        )
    
    # Extract FIXED IO pads only (not PLACED cells)
    io_pads = {}
    comp_match = re.search(r'COMPONENTS\s+\d+\s*;(.*?)END\s+COMPONENTS', content, re.DOTALL)
    if comp_match:
        fixed_pattern = re.compile(
            r'-\s+(\S+)\s+(\S+)\s*.*?FIXED\s*\(\s*(-?\d+)\s+(-?\d+)\s*\)',
            re.DOTALL
        )
        for m in fixed_pattern.finditer(comp_match.group(1)):
            io_pads[m.group(1)] = (int(m.group(3)), int(m.group(4)))
    
    print(f"   [DEF Floorplan] Die area: {die_area}, {len(io_pads)} fixed IO pads")
    return die_area, io_pads


def parse_sdc_file(sdc_path):
    """Parse SDC timing constraints."""
    with open(sdc_path, 'r') as f:
        lines = f.readlines()
    
    constraints = {
        'clock_period': 10.0, 'clock_name': 'clk',
        'input_delays': {}, 'output_delays': {},
        'max_fanout': 10, 'max_transition': 0.5
    }
    
    for line in lines:
        clock_match = re.search(r'create_clock.*-period\s+([\d.]+).*\[get_ports\s+(\w+)\]', line)
        if clock_match:
            constraints['clock_period'] = float(clock_match.group(1))
            constraints['clock_name'] = clock_match.group(2)
        
        in_delay = re.search(r'set_input_delay\s+.*?([\\d.]+).*?\[get_ports\s+(\S+)\]', line)
        if in_delay:
            constraints['input_delays'][in_delay.group(2)] = float(in_delay.group(1))
        
        out_delay = re.search(r'set_output_delay\s+.*?([\\d.]+).*?\[get_ports\s+(\S+)\]', line)
        if out_delay:
            constraints['output_delays'][out_delay.group(2)] = float(out_delay.group(1))
        fanout_match = re.search(r'set_max_fanout\s+([\\d.]+)', line)
        if fanout_match:
            constraints['max_fanout'] = int(float(fanout_match.group(1)))
        trans_match = re.search(r'set_max_transition\s+([\\d.]+)', line)
        if trans_match:
            constraints['max_transition'] = float(trans_match.group(1))
    
    print(f"   [SDC] Clock: {constraints['clock_name']} @ {constraints['clock_period']}ns")
    return constraints


# ═══════════════════════════════════════════════════════════════════
# CONVERTER: Parsed files → 16-feature GNN Data (NO prior placement)
# ═══════════════════════════════════════════════════════════════════

def convert_to_gnn_data_from_netlist(node_attr, pin_attr, cell_lib,
                                      die_area=None, io_pads=None,
                                      sdc_constraints=None):
    """
    Convert parsed Verilog + LEF into the 16-feature PyTorch Geometric Data
    that VLSIPlacementGNN expects.
    
    NO prior placement is used — initial positions are derived from cell
    ordering and connectivity (force-directed initialization).
    
    The GNN will predict the actual (x,y) placement.
    """
    cell_names = list(node_attr[0])
    cell_types = list(node_attr[1])
    num_nodes = len(cell_names)
    
    # ── Determine chip dimensions ──
    if die_area:
        chip_w = die_area[2] - die_area[0]
        chip_h = die_area[3] - die_area[1]
    else:
        # Estimate from total cell area (assume 60% utilization)
        total_cell_area = 0
        for ct in cell_types:
            if ct in cell_lib:
                total_cell_area += cell_lib[ct]['width'] * cell_lib[ct]['height']
        if total_cell_area > 0:
            chip_w = chip_h = np.sqrt(total_cell_area / 0.6)
        else:
            chip_w = chip_h = 1000.0  # Default 1mm × 1mm
    
    # ── Cell sizes from LEF ──
    sizes = np.zeros((num_nodes, 2), dtype=np.float32)
    areas = np.zeros(num_nodes, dtype=np.float32)
    for i, ct in enumerate(cell_types):
        if ct in cell_lib:
            sizes[i, 0] = cell_lib[ct]['width']
            sizes[i, 1] = cell_lib[ct]['height']
        else:
            sizes[i] = [1.0, 1.0]
        areas[i] = sizes[i, 0] * sizes[i, 1]
    
    total_chip_area = chip_w * chip_h + 1e-8
    sizes_norm = sizes / (sizes.max() + 1e-8)
    
    # ── Check chip feasibility ──
    total_cell_area_actual = areas.sum()
    utilization = total_cell_area_actual / total_chip_area
    print(f"\n   ⚠️  CHIP UTILIZATION CHECK:")
    print(f"      Chip area: {chip_w:.1f} × {chip_h:.1f} = {total_chip_area:.1f} µm²")
    print(f"      Total cell area: {total_cell_area_actual:.1f} µm²")
    print(f"      Utilization: {utilization*100:.1f}%")
    
    if utilization > 1.0:
        print(f"      ❌ ERROR: Cells cannot physically fit! ({utilization*100:.0f}% > 100%)")
        print(f"      → Chip too small or too many/large cells")
        print(f"      → Minimum chip size needed: {np.sqrt(total_cell_area_actual/0.8):.0f}×{np.sqrt(total_cell_area_actual/0.8):.0f} µm (80% util)")
        raise ValueError(f"Infeasible placement: chip utilization = {utilization*100:.0f}%")
    elif utilization > 0.85:
        print(f"      ⚠️  WARNING: Very high density! Placement quality may be poor.")
    elif utilization < 0.2:
        print(f"      ℹ️  Low utilization - cells will be spread out")
    else:
        print(f"      ✓ Feasible chip size")
    
    # ── Build edges from netlist connectivity ──
    edge_index, n_orphans, n_nets, cell_connectivity = build_netlist_edges(
        pin_attr, node_attr, cell_names, star_threshold=10
    )
    
    has_netlist = edge_index.shape[1] > 0
    
    if not has_netlist:
        print("   WARNING: No netlist edges, using sequential chain fallback")
        src = np.arange(0, num_nodes - 1, dtype=np.int64)
        dst = np.arange(1, num_nodes, dtype=np.int64)
        edge_index = np.stack([
            np.concatenate([src, dst]),
            np.concatenate([dst, src])
        ], axis=0)
        for idx in range(num_nodes):
            cell_connectivity[idx] = {'pin_count': 0, 'net_count': 0,
                                       'avg_fanout': 0, 'max_fanout': 0}
    
    # ── Initial positions: spread cells across chip area ──
    # Use a simple grid layout as initialization (GNN will predict final)
    grid_cols = max(1, int(np.ceil(np.sqrt(num_nodes * chip_w / chip_h))))
    grid_rows = max(1, int(np.ceil(num_nodes / grid_cols)))
    
    coords = np.zeros((num_nodes, 2), dtype=np.float32)
    for i in range(num_nodes):
        row = i // grid_cols
        col = i % grid_cols
        coords[i, 0] = (col + 0.5) / grid_cols * chip_w
        coords[i, 1] = (row + 0.5) / grid_rows * chip_h
    
    # Place fixed IO pads at their known positions
    if io_pads:
        for pad_name, (px, py) in io_pads.items():
            if pad_name in cell_names:
                idx = cell_names.index(pad_name)
                coords[idx] = [px, py]
    
    # Normalize coordinates to [0, 1]
    coord_min = coords.min(axis=0)
    coord_range = coords.max(axis=0) - coord_min + 1e-8
    coords_norm = (coords - coord_min) / coord_range
    
    # ── Cell classification ──
    is_macro = np.zeros(num_nodes, dtype=np.float32)
    cell_category_arr = np.zeros(num_nodes, dtype=np.float32)
    type_encoding = np.zeros(num_nodes, dtype=np.float32)
    
    unique_types = sorted(set(cell_types))
    type_to_idx = {t: i for i, t in enumerate(unique_types)}
    if len(unique_types) > 1:
        type_encoding = np.array([type_to_idx[t] for t in cell_types], dtype=np.float32) / (len(unique_types) - 1)
    
    for i, (ct, a) in enumerate(zip(cell_types, areas)):
        macro, filler, cat = classify_cell(ct, a)
        is_macro[i] = float(macro)
        cell_category_arr[i] = cat / 3.0
    
    # ── Connectivity features ──
    pin_counts = np.array([cell_connectivity.get(i, {}).get('pin_count', 0) for i in range(num_nodes)], dtype=np.float32)
    net_counts = np.array([cell_connectivity.get(i, {}).get('net_count', 0) for i in range(num_nodes)], dtype=np.float32)
    avg_fanouts = np.array([cell_connectivity.get(i, {}).get('avg_fanout', 0) for i in range(num_nodes)], dtype=np.float32)
    max_fanouts = np.array([cell_connectivity.get(i, {}).get('max_fanout', 0) for i in range(num_nodes)], dtype=np.float32)
    
    pin_counts_norm = pin_counts / (pin_counts.max() + 1e-8)
    net_counts_norm = net_counts / (net_counts.max() + 1e-8)
    avg_fanouts_norm = avg_fanouts / (avg_fanouts.max() + 1e-8)
    max_fanouts_norm = max_fanouts / (max_fanouts.max() + 1e-8)
    
    # ── Additional features ──
    relative_area = areas / total_chip_area
    relative_area_norm = relative_area / (relative_area.max() + 1e-8)
    
    aspect_ratio = sizes[:, 0] / (sizes[:, 1] + 1e-8)
    aspect_ratio = np.clip(aspect_ratio, 0, 10)
    aspect_ratio_norm = aspect_ratio / (aspect_ratio.max() + 1e-8)
    
    conn_importance = pin_counts * net_counts
    conn_importance_norm = conn_importance / (conn_importance.max() + 1e-8)
    
    neighbor_area_avg = np.zeros(num_nodes, dtype=np.float32)
    if edge_index.shape[1] > 0:
        ei = edge_index
        for e in range(ei.shape[1]):
            s, d = ei[0, e], ei[1, e]
            neighbor_area_avg[s] += areas[d]
        degree = np.bincount(ei[0], minlength=num_nodes).astype(np.float32)
        degree[degree == 0] = 1
        neighbor_area_avg /= degree
    neighbor_area_norm = neighbor_area_avg / (neighbor_area_avg.max() + 1e-8)
    
    # ── Assemble 16-feature vector (same as training) ──
    node_features = np.zeros((num_nodes, 16), dtype=np.float32)
    node_features[:, 0]  = coords_norm[:, 0]       # [0] x position (initial)
    node_features[:, 1]  = coords_norm[:, 1]       # [1] y position (initial)
    node_features[:, 2]  = sizes_norm[:, 0]         # [2] width
    node_features[:, 3]  = sizes_norm[:, 1]         # [3] height
    node_features[:, 4]  = np.log1p(areas)          # [4] log area
    node_features[:, 4] /= (node_features[:, 4].max() + 1e-8)
    node_features[:, 5]  = type_encoding            # [5] cell type
    node_features[:, 6]  = is_macro                 # [6] is_macro
    node_features[:, 7]  = cell_category_arr        # [7] cell category
    node_features[:, 8]  = pin_counts_norm          # [8] pin count
    node_features[:, 9]  = net_counts_norm          # [9] net count
    node_features[:, 10] = avg_fanouts_norm         # [10] avg fanout
    node_features[:, 11] = max_fanouts_norm         # [11] max fanout
    node_features[:, 12] = relative_area_norm       # [12] relative area
    node_features[:, 13] = aspect_ratio_norm        # [13] aspect ratio
    node_features[:, 14] = neighbor_area_norm       # [14] avg neighbor size
    node_features[:, 15] = conn_importance_norm     # [15] connectivity importance
    
    # ── Build Data object ──
    data = Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        y=torch.zeros(num_nodes, 2)   # no ground truth — GNN predicts this
    )
    data.num_cells = num_nodes
    data.sample_name = "user_design"
    data.design_name = "user_design"
    data.original_coords = torch.tensor(coords, dtype=torch.float)
    data.is_macro = torch.tensor(is_macro, dtype=torch.float)
    data.areas = torch.tensor(areas, dtype=torch.float)
    data.cell_names = cell_names
    data.cell_types = cell_types
    data.chip_w = chip_w
    data.chip_h = chip_h
    
    print(f"\n   Converted to GNN format:")
    print(f"      Cells: {num_nodes:,}")
    print(f"      Edges: {edge_index.shape[1]:,}")
    print(f"      Macros: {int(is_macro.sum())}")
    print(f"      Features: 16 per cell")
    
    return data


# ═══════════════════════════════════════════════════════════════════════
# GUI: Browse files → Parse → GNN Inference → Legalize → Visualize
# ═══════════════════════════════════════════════════════════════════════

def run_gnn_inference_gui():
    """
    Tkinter GUI to:
      1. Browse for Verilog, LEF (required), Model (required),
         and optionally SDC, Floorplan DEF
      2. Parse all files → build 16-feature graph
      3. Run GNN inference (predicted placement)
      4. Legalize placement (snap to rows/sites)
      5. Export DEF + JSON and visualize
    """
    file_paths = {
        'verilog': None, 'lef': None, 'model': None,
        'sdc': None, 'def': None,
    }
    
    root = tk.Tk()
    root.title("VLSI GNN Placement — Inference from Netlist")
    root.geometry("750x580")
    root.configure(bg='#1e1e2e')
    root.resizable(False, False)
    
    style = ttk.Style()
    style.theme_use('clam')
    style.configure('H.TLabel', background='#1e1e2e', foreground='#cdd6f4',
                    font=('Segoe UI', 15, 'bold'))
    style.configure('Sub.TLabel', background='#1e1e2e', foreground='#a6adc8',
                    font=('Segoe UI', 9))
    style.configure('B.TButton', font=('Segoe UI', 9))
    
    ttk.Label(root, text="VLSI GNN Placement Inference", style='H.TLabel').pack(pady=(16, 2))
    ttk.Label(root, text="GNN predicts placement from Verilog + LEF — no existing placement needed.",
              style='Sub.TLabel').pack(pady=(0, 12))
    
    file_frame = tk.Frame(root, bg='#1e1e2e')
    file_frame.pack(fill='x', padx=28)
    
    path_vars = {}
    
    file_defs = [
        ('verilog', 'Verilog Netlist (.v)',     [('Verilog', '*.v'), ('All', '*.*')],     True),
        ('lef',     'LEF Cell Library (.lef)',   [('LEF', '*.lef'), ('All', '*.*')],       True),
        ('model',   'Model Weights (.pth)',      [('PyTorch', '*.pth'), ('All', '*.*')],   True),
        ('sdc',     'SDC Constraints (.sdc)',    [('SDC', '*.sdc'), ('All', '*.*')],        False),
        ('def',     'Floorplan DEF (.def)',      [('DEF', '*.def'), ('All', '*.*')],        False),
    ]
    
    for key, label_text, ftypes, required in file_defs:
        row = tk.Frame(file_frame, bg='#1e1e2e')
        row.pack(fill='x', pady=3)
        
        tag = " *" if required else ""
        tk.Label(row, text=f"{label_text}{tag}", bg='#1e1e2e', fg='#cdd6f4',
                 font=('Segoe UI', 10), width=28, anchor='w').pack(side='left')
        
        pv = tk.StringVar(value="No file selected")
        tk.Label(row, textvariable=pv, bg='#313244', fg='#bac2de',
                 font=('Consolas', 9), width=38, anchor='w',
                 relief='flat', padx=6, pady=3).pack(side='left', padx=(4, 6))
        path_vars[key] = pv
        
        def make_browse(k=key, ft=ftypes, v=pv):
            def browse():
                p = filedialog.askopenfilename(filetypes=ft)
                if p:
                    file_paths[k] = p
                    v.set(os.path.basename(p))
            return browse
        
        ttk.Button(row, text="Browse", command=make_browse(), style='B.TButton').pack(side='left')
    
    ttk.Label(file_frame, text="* = required      (DEF is for die area / IO pads only — NOT cell placement)",
              style='Sub.TLabel').pack(anchor='w', pady=(6, 0))
    
    status_var = tk.StringVar(value="Ready — select files and click Run Inference")
    tk.Label(root, textvariable=status_var, bg='#1e1e2e', fg='#a6e3a1',
             font=('Segoe UI', 10), wraplength=680, justify='left').pack(
        pady=(14, 4), padx=28, anchor='w')
    
    progress = ttk.Progressbar(root, mode='determinate', length=690)
    progress.pack(pady=(2, 10), padx=28)
    
    def run_inference():
        # Validate required files
        for key in ['verilog', 'lef', 'model']:
            if not file_paths[key]:
                nice = {'verilog': 'Verilog Netlist', 'lef': 'LEF Library',
                        'model': 'Model Weights'}[key]
                messagebox.showerror("Missing File", f"Please select a {nice} file.")
                return
        
        progress['value'] = 0
        root.update_idletasks()
        
        try:
            # ── Step 1: Parse Verilog ──
            status_var.set("Step 1/6 — Parsing Verilog netlist...")
            progress['value'] = 8; root.update_idletasks()
            node_attr, pin_attr, cell_names, cell_types = parse_verilog_netlist(
                file_paths['verilog'])
            
            # ── Step 2: Parse LEF ──
            status_var.set("Step 2/6 — Parsing LEF cell library...")
            progress['value'] = 20; root.update_idletasks()
            cell_lib = parse_lef_file(file_paths['lef'])
            
            # ── Step 3: Parse Floorplan DEF (optional) ──
            die_area, io_pads = None, None
            if file_paths['def']:
                status_var.set("Step 3/6 — Parsing floorplan DEF (die area + IO pads)...")
                progress['value'] = 28; root.update_idletasks()
                die_area, io_pads = parse_floorplan_def(file_paths['def'])
            
            # ── Step 4: Parse SDC (optional) ──
            sdc_constraints = None
            if file_paths['sdc']:
                status_var.set("Step 4/6 — Parsing SDC constraints...")
                progress['value'] = 32; root.update_idletasks()
                sdc_constraints = parse_sdc_file(file_paths['sdc'])
            
            # ── Step 5: Convert to GNN format ──
            status_var.set("Step 5/6 — Building 16-feature graph...")
            progress['value'] = 40; root.update_idletasks()
            
            gnn_data = convert_to_gnn_data_from_netlist(
                node_attr, pin_attr, cell_lib,
                die_area=die_area, io_pads=io_pads,
                sdc_constraints=sdc_constraints
            )
            
            # ═══════════════════════════════════════════════════════════════
            # SHOW EXTRACTED FEATURES — Display what GNN will receive
            # ═══════════════════════════════════════════════════════════════
            progress['value'] = 45; root.update_idletasks()
            
            # Create feature display window
            feature_window = tk.Toplevel(root)
            feature_window.title("Extracted Features — GNN Input Data")
            feature_window.geometry("900x700")
            feature_window.configure(bg='#1e1e2e')
            feature_window.transient(root)
            feature_window.grab_set()
            
            # Main container with scrollbar
            main_frame = tk.Frame(feature_window, bg='#1e1e2e')
            main_frame.pack(fill='both', expand=True, padx=15, pady=15)
            
            # Canvas + Scrollbar
            canvas = tk.Canvas(main_frame, bg='#1e1e2e', highlightthickness=0)
            scrollbar = tk.Scrollbar(main_frame, orient="vertical", command=canvas.yview)
            scrollable_frame = tk.Frame(canvas, bg='#1e1e2e')
            
            scrollable_frame.bind(
                "<Configure>",
                lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
            )
            
            canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
            canvas.configure(yscrollcommand=scrollbar.set)
            
            # Title
            tk.Label(scrollable_frame, text="📊 Extracted Features from Input Files", 
                    bg='#1e1e2e', fg='#89b4fa', 
                    font=('Segoe UI', 16, 'bold')).pack(pady=(0, 10))
            
            tk.Label(scrollable_frame, text="Features below will be input to the GNN model for placement prediction",
                    bg='#1e1e2e', fg='#a6adc8', 
                    font=('Segoe UI', 9, 'italic')).pack(pady=(0, 15))
            
            # ── Section 1: FROM VERILOG NETLIST ──
            tk.Label(scrollable_frame, text="━" * 85, bg='#1e1e2e', fg='#45475a',
                    font=('Courier', 8)).pack()
            tk.Label(scrollable_frame, text="📄 FROM VERILOG NETLIST (.v)", 
                    bg='#1e1e2e', fg='#f9e2af', 
                    font=('Segoe UI', 12, 'bold')).pack(anchor='w', pady=(8, 5))
            
            verilog_info = f"""   ✓ Cell instances: {len(cell_names):,} cells
   ✓ Cell types: {len(set(cell_types))} unique types  (e.g., {', '.join(list(set(cell_types))[:5])})
   ✓ Nets: {len(set(pin_attr[0])) if pin_attr is not None else 0:,} connections parsed
   ✓ Connectivity: Pin-to-net mapping extracted
   
   → Extracted Features:
      [5]  Cell type encoding
      [6]  Is macro flag (PLL, SRAM detection)
      [7]  Cell category (filler/combo/seq/macro)
      [8]  Pin count per cell
      [9]  Net count per cell
      [10] Average net fanout
      [11] Maximum net fanout
      [15] Connectivity importance (pins × nets)"""
            
            tk.Label(scrollable_frame, text=verilog_info, 
                    bg='#1e1e2e', fg='#cdd6f4', 
                    font=('Consolas', 9), justify='left').pack(anchor='w', padx=20)
            
            # ── Section 2: FROM LEF FILE ──
            tk.Label(scrollable_frame, text="━" * 85, bg='#1e1e2e', fg='#45475a',
                    font=('Courier', 8)).pack(pady=(15, 0))
            tk.Label(scrollable_frame, text="📐 FROM LEF LIBRARY (.lef)", 
                    bg='#1e1e2e', fg='#a6e3a1', 
                    font=('Segoe UI', 12, 'bold')).pack(anchor='w', pady=(8, 5))
            
            lef_info = f"""   ✓ Cell library: {len(cell_lib)} cell definitions
   ✓ Physical dimensions extracted for each cell type
   ✓ Sample cells: {', '.join(list(cell_lib.keys())[:4])}...
   
   → Extracted Features:
      [2]  Width (from LEF SIZE statement)
      [3]  Height (from LEF SIZE statement)
      [4]  Log area (calculated: width × height)
      [12] Relative area (cell area / chip area)
      [13] Aspect ratio (width / height)"""
            
            tk.Label(scrollable_frame, text=lef_info, 
                    bg='#1e1e2e', fg='#cdd6f4', 
                    font=('Consolas', 9), justify='left').pack(anchor='w', padx=20)
            
            # ── Section 3: COMPUTED FEATURES ──
            tk.Label(scrollable_frame, text="━" * 85, bg='#1e1e2e', fg='#45475a',
                    font=('Courier', 8)).pack(pady=(15, 0))
            tk.Label(scrollable_frame, text="🧮 COMPUTED FEATURES", 
                    bg='#1e1e2e', fg='#cba6f7', 
                    font=('Segoe UI', 12, 'bold')).pack(anchor='w', pady=(8, 5))
            
            computed_info = f"""   ✓ Initial positions: Grid layout initialization
   ✓ Neighbor context: Average size of connected cells
   
   → Extracted Features:
      [0]  X position (grid-based initialization)
      [1]  Y position (grid-based initialization)
      [14] Average neighbor size (from connectivity graph)"""
            
            tk.Label(scrollable_frame, text=computed_info, 
                    bg='#1e1e2e', fg='#cdd6f4', 
                    font=('Consolas', 9), justify='left').pack(anchor='w', padx=20)
            
            # ── Section 4: COMPLETE 16-FEATURE VECTOR ──
            tk.Label(scrollable_frame, text="━" * 85, bg='#1e1e2e', fg='#45475a',
                    font=('Courier', 8)).pack(pady=(15, 0))
            tk.Label(scrollable_frame, text="🎯 COMPLETE 16-FEATURE VECTOR (Input to GNN)", 
                    bg='#1e1e2e', fg='#f38ba8', 
                    font=('Segoe UI', 12, 'bold')).pack(anchor='w', pady=(8, 5))
            
            # Get features for first 5 cells
            num_cells_to_show = min(5, gnn_data.num_cells)
            
            feature_names = [
                "[0]  X position (initial)",
                "[1]  Y position (initial)", 
                "[2]  Cell width",
                "[3]  Cell height",
                "[4]  Log area (normalized)",
                "[5]  Cell type (encoded)",
                "[6]  Is macro (0/1)",
                "[7]  Cell category (0-1)",
                "[8]  Pin count",
                "[9]  Net count",
                "[10] Avg net fanout",
                "[11] Max net fanout",
                "[12] Relative area",
                "[13] Aspect ratio",
                "[14] Avg neighbor size",
                "[15] Connectivity importance"
            ]
            
            sources = [
                "Computed", "Computed", "LEF", "LEF", "LEF+Calc",
                "Verilog", "Verilog", "Verilog", "Verilog", "Verilog",
                "Verilog", "Verilog", "LEF+Calc", "LEF+Calc", "Graph+LEF", "Verilog+Calc"
            ]
            
            feature_table = f"   Showing first {num_cells_to_show} cells (out of {gnn_data.num_cells} total)\n\n"
            
            for cell_idx in range(num_cells_to_show):
                sample_features = gnn_data.x[cell_idx].cpu().numpy()
                is_macro_flag = "🔴 MACRO" if gnn_data.is_macro[cell_idx] > 0.5 else "🔵 STD"
                
                feature_table += f"\n   {'-'*80}\n"
                feature_table += f"   CELL #{cell_idx+1}: '{gnn_data.cell_names[cell_idx]}' (type: {gnn_data.cell_types[cell_idx]}) {is_macro_flag}\n"
                feature_table += f"   {'-'*80}\n\n"
                feature_table += f"   {'Feature':<35} {'Value':>10}  {'Source'}\n"
                feature_table += f"   {'─'*35} {'─'*10}  {'─'*20}\n"
                
                for i, (fname, fval, src) in enumerate(zip(feature_names, sample_features, sources)):
                    feature_table += f"   {fname:<35} {fval:>10.4f}  {src}\n"
                
                feature_table += "\n"
            
            tk.Label(scrollable_frame, text=feature_table, 
                    bg='#1e1e2e', fg='#cdd6f4', 
                    font=('Consolas', 8), justify='left').pack(anchor='w', padx=20)
            
            # ── Section 5: GRAPH STATISTICS ──
            tk.Label(scrollable_frame, text="━" * 85, bg='#1e1e2e', fg='#45475a',
                    font=('Courier', 8)).pack(pady=(15, 0))
            tk.Label(scrollable_frame, text="📈 GRAPH STATISTICS", 
                    bg='#1e1e2e', fg='#74c7ec', 
                    font=('Segoe UI', 12, 'bold')).pack(anchor='w', pady=(8, 5))
            
            n_macros = int(gnn_data.is_macro.sum())
            avg_degree = gnn_data.edge_index.shape[1] / gnn_data.num_cells
            
            stats_info = f"""   Total Cells: {gnn_data.num_cells:,}
   Macros: {n_macros} ({n_macros/gnn_data.num_cells*100:.1f}%)
   Standard Cells: {gnn_data.num_cells - n_macros:,} ({(gnn_data.num_cells-n_macros)/gnn_data.num_cells*100:.1f}%)
   
   Graph Connectivity:
   • Edges: {gnn_data.edge_index.shape[1]:,} directed edges
   • Average degree: {avg_degree:.1f} connections per cell
   • Chip dimensions: {gnn_data.chip_w:.1f} × {gnn_data.chip_h:.1f} µm
   
   ✓ All 16 features extracted successfully!
   ✓ Ready for GNN inference"""
            
            tk.Label(scrollable_frame, text=stats_info, 
                    bg='#1e1e2e', fg='#cdd6f4', 
                    font=('Consolas', 9), justify='left').pack(anchor='w', padx=20, pady=(0, 10))
            
            # Continue button
            def close_feature_window():
                feature_window.destroy()
            
            tk.Label(scrollable_frame, text="━" * 85, bg='#1e1e2e', fg='#45475a',
                    font=('Courier', 8)).pack(pady=(10, 15))
            
            tk.Button(scrollable_frame, text="✓ Continue to GNN Inference", 
                     command=close_feature_window,
                     bg='#89b4fa', fg='#1e1e2e', font=('Segoe UI', 11, 'bold'),
                     relief='flat', padx=40, pady=10, cursor='hand2',
                     activebackground='#74c7ec').pack(pady=(5, 15))
            
            canvas.pack(side="left", fill="both", expand=True)
            scrollbar.pack(side="right", fill="y")
            
            # Wait for user to close the window
            root.wait_window(feature_window)
            
            # ── Step 6: Load model & run inference ──
            status_var.set("Step 6/6 — Loading model & running GNN inference...")
            progress['value'] = 55; root.update_idletasks()
            
            inference_model = VLSIPlacementGNN(
                input_dim=16, hidden_dim=128, output_dim=2,
                num_layers=4, heads=4
            ).to(device)
            
            ckpt = torch.load(file_paths['model'], map_location=device)
            inference_model.load_state_dict(ckpt['model_state_dict'])
            inference_model.eval()
            
            # Keep CPU copies of tensors we need for visualization BEFORE .to(device)
            is_macro_cpu = gnn_data.is_macro.clone().cpu()
            edge_index_cpu = gnn_data.edge_index.clone().cpu()
            
            with torch.no_grad():
                gnn_data_dev = gnn_data.to(device)
                predictions = inference_model(gnn_data_dev).cpu().numpy()
            
            progress['value'] = 75; root.update_idletasks()
            
            # ── Scale predictions to real chip coordinates ──
            chip_w = gnn_data.chip_w
            chip_h = gnn_data.chip_h
            pred_coords = predictions * np.array([chip_w, chip_h])
            
            # ── Legalize placement ──
            status_var.set("Legalizing placement (row/site snapping)...")
            progress['value'] = 82; root.update_idletasks()
            
            legalized = legalize_placement(
                pred_coords, chip_width_um=chip_w, chip_height_um=chip_h
            )
            
            # ── Save results ──
            status_var.set("Saving results...")
            progress['value'] = 90; root.update_idletasks()
            
            output_dir = os.path.dirname(file_paths['verilog'])
            
            # Output DEF (the whole point — GNN-generated placement)
            def_out = os.path.join(output_dir, "gnn_predicted_placement.def")
            with open(def_out, 'w') as f:
                f.write("VERSION 5.8 ;\n")
                if die_area:
                    f.write(f"DIEAREA ( {die_area[0]} {die_area[1]} )"
                            f" ( {die_area[2]} {die_area[3]} ) ;\n")
                f.write(f"DESIGN {gnn_data.design_name} ;\n\n")
                f.write(f"COMPONENTS {gnn_data.num_cells} ;\n")
                for idx in range(len(legalized)):
                    x_int = int(legalized[idx, 0])
                    y_int = int(legalized[idx, 1])
                    f.write(f"  - {gnn_data.cell_names[idx]} {gnn_data.cell_types[idx]}"
                            f" + PLACED ( {x_int} {y_int} ) N ;\n")
                f.write("END COMPONENTS\n\nEND DESIGN\n")
            
            # Output JSON
            json_out = os.path.join(output_dir, "gnn_predicted_placement.json")
            result = {
                'design': gnn_data.design_name,
                'num_cells': int(gnn_data.num_cells),
                'chip_width': float(chip_w),
                'chip_height': float(chip_h),
                'cells': [
                    {
                        'name': gnn_data.cell_names[i],
                        'type': gnn_data.cell_types[i],
                        'predicted_x': float(legalized[i, 0]),
                        'predicted_y': float(legalized[i, 1]),
                        'global_x': float(pred_coords[i, 0]),
                        'global_y': float(pred_coords[i, 1]),
                    }
                    for i in range(len(legalized))
                ]
            }
            with open(json_out, 'w') as f:
                json.dump(result, f, indent=2)
            
            progress['value'] = 100
            status_var.set(f"Done! Saved: {os.path.basename(def_out)}, {os.path.basename(json_out)}")
            root.update_idletasks()
            
            # ── Close GUI → visualize in matplotlib ──
            root.destroy()
            
            print("\n" + "=" * 70)
            print("GNN PLACEMENT INFERENCE COMPLETE")
            print("=" * 70)
            print(f"   Cells: {gnn_data.num_cells:,}")
            print(f"   Chip:  {chip_w:.1f} x {chip_h:.1f}")
            print(f"   DEF:   {def_out}")
            print(f"   JSON:  {json_out}")
            print("=" * 70)
            
            # Plot: Global vs Legalized  (use CPU copies saved earlier)
            fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=120)
            fig.patch.set_facecolor('#1a1a1a')
            
            macro_mask = is_macro_cpu.numpy().astype(bool)
            
            for ax, coords_plot, title, color in [
                (axes[0], pred_coords, 'GNN Global Placement (Raw)', 'cyan'),
                (axes[1], legalized,  'Legalized Placement (Row-Aligned)', 'lime'),
            ]:
                ax.set_facecolor('#1a1a1a')
                ax.scatter(coords_plot[~macro_mask, 0], coords_plot[~macro_mask, 1],
                          s=3, c=color, alpha=0.6, label='Standard cells')
                if macro_mask.any():
                    ax.scatter(coords_plot[macro_mask, 0], coords_plot[macro_mask, 1],
                              s=40, c='red', marker='s', alpha=0.9, label='Macros')
                
                # Sample edges
                ei = edge_index_cpu.numpy()
                for e in range(0, min(600, ei.shape[1]), 5):
                    s, d = ei[0, e], ei[1, e]
                    ax.plot([coords_plot[s, 0], coords_plot[d, 0]],
                            [coords_plot[s, 1], coords_plot[d, 1]],
                            'yellow', alpha=0.1, linewidth=0.3)
                
                ax.set_title(title, fontsize=14, color='white', pad=12)
                ax.set_xlabel('X (µm)', color='white')
                ax.set_ylabel('Y (µm)', color='white')
                ax.tick_params(colors='white')
                ax.legend(fontsize=9, loc='upper right')
                ax.grid(True, alpha=0.15, color='gray')
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'gnn_placement_result.png'),
                        dpi=150, facecolor='#1a1a1a', bbox_inches='tight')
            plt.show()
            
            # Displacement stats
            disp = np.linalg.norm(legalized - pred_coords, axis=1)
            print(f"\nLegalization displacement:")
            print(f"   Avg: {np.mean(disp):.2f} µm")
            print(f"   Max: {np.max(disp):.2f} µm")
            print(f"   Cells moved: {np.sum(disp > 0.1)}/{len(disp)}")
        
        except Exception as e:
            import traceback
            traceback.print_exc()
            # GUI may already be destroyed — guard widget access
            try:
                status_var.set(f"ERROR: {str(e)}")
                progress['value'] = 0
                messagebox.showerror("Error", f"An error occurred:\n\n{str(e)}")
            except tk.TclError:
                print(f"\nERROR: {str(e)}")
    
    run_btn = tk.Button(root, text="▶  Run Inference", command=run_inference,
                        bg='#89b4fa', fg='#1e1e2e', font=('Segoe UI', 12, 'bold'),
                        relief='flat', padx=30, pady=8, cursor='hand2',
                        activebackground='#74c7ec', activeforeground='#1e1e2e')
    run_btn.pack(pady=(6, 16))
    
    root.mainloop()


# ── Launch ──
print("Launching GNN Placement Inference GUI...")
print("   Required: Verilog netlist (.v) + LEF library (.lef) + Model (.pth)")
print("   Optional: SDC (.sdc) + Floorplan DEF (.def, for die area/IO pads only)")
print("   The GNN predicts placement from scratch — no existing placement needed.\n")
run_gnn_inference_gui()

Launching GNN Placement Inference GUI...
   Required: Verilog netlist (.v) + LEF library (.lef) + Model (.pth)
   Optional: SDC (.sdc) + Floorplan DEF (.def, for die area/IO pads only)
   The GNN predicts placement from scratch — no existing placement needed.

